In [0]:
# Databricks notebook source
# MAGIC %pip install geopy
# MAGIC dbutils.library.restartPython()

# COMMAND ----------

# MAGIC %md
# MAGIC # 02 — Silver Layer: Clean, Parse & Enrich (v8 — PDF Contract Enforcement)
# MAGIC
# MAGIC **Input**: `virtue_foundation.ghana.bronze_facilities_raw`
# MAGIC **Output**: `virtue_foundation.ghana.silver_facilities_cleaned`
# MAGIC
# MAGIC ## What changed in v8 (vs v7):
# MAGIC
# MAGIC | Fix | Description |
# MAGIC |-----|-------------|
# MAGIC | v8-1  | `facilityTypeId` enum hard-mapped back to PDF set: hospital/pharmacy/doctor/clinic/dentist only |
# MAGIC | v8-2  | Internal extended types (chps, eye_clinic, maternity_home, diagnostic_center) kept in `facility_type_clean` only |
# MAGIC | v8-3  | `Ghana → Greater Accra` REMOVED from region fallback — country-only evidence stays Unknown |
# MAGIC | v8-4  | Kasoa hard override BEFORE any text-signal fallback — cannot be overridden by broader text |
# MAGIC | v8-5  | `phone_numbers` canonical field rewritten to E.164 normalized list (not just official_phone) |
# MAGIC | v8-6  | `organizationdescription` neutrality pass: strips subjective/religious/promotional wording |
# MAGIC | v8-7  | `countries_parsed` ISO alpha-2 validation + enforce GH for Ghana records |
# MAGIC | v8-8  | Second numeric inference layer: rule-based from facility type + clinical language |
# MAGIC | v8-9  | Numeric confidence columns: doctors_confidence, capacity_confidence, year_confidence |
# MAGIC | v8-10 | Capability whitelist: ICU/NICU/trauma/24h-emergency never filtered |
# MAGIC | v8-11 | Equipment context-inference: laboratory→lab equipment, ICU→ventilator/oxygen, etc. |
# MAGIC | v8-12 | Anomaly detection flags: hospital_low_capacity, pharmacy_claims_icu, region_city_mismatch, etc. |
# MAGIC | v8-13 | Stronger dedup: normalise legal suffixes, punctuation, whitespace before clustering |
# MAGIC | v8-14 | Canonical field overwrite: numberDoctors/capacity/yearestablished/acceptsvolunteers use enriched values |
# MAGIC | v8-15 | `operatorTypeId` validated to PDF enum: public/private only |
# MAGIC | v8-16 | Contract validation block before write: enum, required fields, specialty vocab, neutral descriptions |
# MAGIC | v8-17 | Specialty vocabulary normalization against FDR canonical camelCase taxonomy |

# COMMAND ----------
# MAGIC %md ## 0. Imports & Config

# COMMAND ----------

import json
import re
import time
from typing import Optional, List, Dict, Tuple
from datetime import datetime, timezone
from functools import reduce

import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StringType, IntegerType, FloatType, BooleanType,
    ArrayType, DoubleType, StructType, StructField, LongType,
)
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf

spark = SparkSession.builder.getOrCreate()
print(f"Run: {datetime.now(timezone.utc).isoformat()}")
print(f"Spark: {spark.version}")

# COMMAND ----------

class Config:
    BRONZE         = "virtue_foundation.ghana.bronze_facilities_raw"
    SILVER         = "virtue_foundation.ghana.silver_facilities_cleaned"
    EXTRACTION_VER = "silver_v8"

    # Ghana centroid (last-resort fallback only — never used for region assignment)
    DEFAULT_LAT = 7.9465
    DEFAULT_LON = -1.0232

    # Extraction plausibility bounds
    MAX_DOCTORS = 500
    MAX_BEDS    = 5000
    MAX_AREA    = 100_000
    MIN_YEAR    = 1850
    MAX_YEAR    = 2026

cfg = Config()
spark.sql("CREATE DATABASE IF NOT EXISTS virtue_foundation.ghana")

# ── PDF canonical enums (v8: enforced at write time) ──────────────────────────
# facilityTypeId: ONLY these 5 values allowed in canonical field
PDF_FACILITY_TYPE_ENUM = frozenset({"hospital", "pharmacy", "doctor", "clinic", "dentist"})

# Internal extended types → PDF enum mapping
EXTENDED_TO_PDF_ENUM: Dict[str, str] = {
    "hospital":          "hospital",
    "clinic":            "clinic",
    "pharmacy":          "pharmacy",
    "dentist":           "dentist",
    "doctor":            "doctor",
    # Extended types → nearest PDF enum
    "chps":              "clinic",
    "maternity_home":    "clinic",
    "eye_clinic":        "clinic",
    "diagnostic_center": "clinic",
}

# operatorTypeId: ONLY these 2 values
PDF_OPERATOR_TYPE_ENUM = frozenset({"public", "private"})

# FDR canonical specialty taxonomy (exact camelCase)
FDR_SPECIALTY_VOCAB = frozenset({
    "internalMedicine", "familyMedicine", "pediatrics", "cardiology",
    "generalSurgery", "emergencyMedicine", "gynecologyAndObstetrics",
    "orthopedicSurgery", "dentistry", "ophthalmology", "radiology",
    "pathology", "infectiousDiseases", "nephrology", "criticalCareMedicine",
    "cardiacSurgery", "plasticSurgery", "neurology", "psychiatry",
    "anesthesia", "dermatology", "urology", "gastroenterology",
    "pulmonology", "endocrinologyAndDiabetesAndMetabolism",
    "neonatologyPerinatalMedicine", "medicalOncology",
    "physicalMedicineAndRehabilitation", "otolaryngology",
    "geriatricsInternalMedicine", "hospiceAndPalliativeInternalMedicine",
    "publicHealth", "globalHealthAndInternationalHealth",
    "clinicalPathology", "obstetricsAndMaternityCare",
    "reproductiveEndocrinologyAndInfertility",
    "maternalFetalMedicineOrPerinatology", "socialAndBehavioralSciences",
    "orthodontics", "familyPlanningAndComplexContraception",
})

# COMMAND ----------
# MAGIC %md ## 1. Read Bronze

# COMMAND ----------

bronze = spark.table(cfg.BRONZE)
print(f"Bronze rows    : {bronze.count():,}")
print(f"Bronze columns : {len(bronze.columns)}")
print("Bronze column names:")
for c in bronze.columns:
    print(f"  {c}")

# COMMAND ----------
# MAGIC %md ## 2. JSON Array Parser — v7 (unchanged, all smoke tests pass)

# COMMAND ----------

def _safe_json_parse(text: Optional[str]) -> List[str]:
    """Parse JSON arrays that may have double-escaped quotes from CSV export."""
    if text is None:
        return []
    text = str(text).strip()
    if text in ("", "null", "[]", "None", "nan"):
        return []

    text_unescaped_once = text.replace('""', '"').strip()
    if text_unescaped_once in ("", "[]", "null", "None", '"[]"', "'[]'"):
        return []

    attempts: List[str] = []
    if '""' in text:
        attempts.append(text.replace('""', '"'))
    if text.startswith('"[') and text.endswith(']"'):
        attempts.append(text[1:-1])
    attempts.append(text)

    for attempt in attempts:
        attempt = attempt.strip()
        if attempt in ("", "[]", "null", "None", '"[]"'):
            return []
        try:
            result = json.loads(attempt)
            if isinstance(result, list):
                return [str(x).strip() for x in result
                        if x is not None and str(x).strip() not in ("", "null", "None")]
            elif isinstance(result, dict):
                return [json.dumps(result)]
            else:
                s = str(result).strip()
                if s.startswith("[") and s not in ("[]",):
                    try:
                        inner = json.loads(s)
                        if isinstance(inner, list):
                            return [str(x).strip() for x in inner
                                    if x is not None and str(x).strip() not in ("", "null", "None")]
                    except Exception:
                        pass
                return [s] if s and s not in ("null", "None", "[]") else []
        except (json.JSONDecodeError, ValueError):
            pass

    cleaned = text.strip('"').strip("'").strip()
    if not cleaned or cleaned in ("null", "None", "[]"):
        return []

    if (
        "," in cleaned
        and len(cleaned) < 500
        and cleaned.count(".") <= 2
        and not cleaned.count("\n")
    ):
        items = [x.strip().strip('"').strip("'") for x in cleaned.split(",")]
        items = [i for i in items if i and i not in ("null", "None", "[]")]
        if items and all(len(i) < 120 for i in items) and len(items) <= 20:
            return items

    if cleaned and cleaned not in ("null", "None", "[]") and len(cleaned) < 200:
        return [cleaned]

    return []


@pandas_udf(ArrayType(StringType()))
def safe_json_udf(col: pd.Series) -> pd.Series:
    return col.apply(_safe_json_parse)


_tests = [
    ('[\"infectiousDiseases\",\"pediatrics\"]', 2),
    ('[]', 0), ('null', 0), (None, 0), ('""[]""', 0),
    ('"[]"', 0), ('[\"single\"]', 1), ('plain text', 1),
    ('["+233249354576","+233203928883"]', 2),
]
print("JSON parser smoke tests:")
all_passed = True
for val, expected in _tests:
    got    = len(_safe_json_parse(val))
    status = "✅" if got == expected else f"❌ expected {expected}, got {got}"
    if got != expected: all_passed = False
    print(f"  {status}  {repr(str(val)[:40]):44} → {got} items")
print(f"{'All tests passed ✅' if all_passed else '⚠️  Some tests FAILED'}")

# COMMAND ----------
# MAGIC %md ## 3. E.164 Phone Normalization

# COMMAND ----------

def _normalize_e164(phone: Optional[str]) -> Optional[str]:
    if not phone:
        return None
    s = str(phone).strip().strip('"').strip("'").strip()
    if not s or s.lower() in ("null", "none", "nan", ""):
        return None
    s_clean = re.sub(r"[\s\-\.\(\)\/]", "", s)
    if s_clean.startswith("+"):
        digits = s_clean[1:]
        if re.fullmatch(r"\d{7,15}", digits):
            return s_clean
        return None
    if not re.fullmatch(r"\d+", s_clean):
        return None
    d = s_clean
    if len(d) == 10 and d.startswith("0") and d[1] in "23456789":
        return "+233" + d[1:]
    if len(d) == 9 and d[0] in "23456789":
        return "+233" + d
    if len(d) == 12 and d.startswith("233"):
        return "+" + d
    if 7 <= len(d) <= 15:
        return "+" + d
    return None


def _normalize_phone_list(raw_items: Optional[List]) -> List[str]:
    if not raw_items:
        return []
    seen: Dict[str, str] = {}
    for raw in raw_items:
        norm = _normalize_e164(str(raw) if raw is not None else None)
        if norm and norm not in seen:
            seen[norm] = norm
    return list(seen.keys())


@pandas_udf(ArrayType(StringType()))
def normalize_phone_list_udf(phones: pd.Series) -> pd.Series:
    result = []
    for items in phones:
        lst = list(items) if items is not None else []
        result.append(_normalize_phone_list(lst))
    return pd.Series(result)


@pandas_udf(StringType())
def extract_official_phone_udf(phones: pd.Series) -> pd.Series:
    result = []
    for items in phones:
        lst = list(items) if items is not None else []
        if not lst:
            result.append(None)
            continue
        gh = [p for p in lst if str(p).startswith("+233")]
        result.append(gh[0] if gh else lst[0])
    return pd.Series(result)


# v8: Rewrite canonical `phone_numbers` STRING field to normalized JSON array
@pandas_udf(StringType())
def normalized_phone_numbers_string_udf(phones: pd.Series) -> pd.Series:
    """Rewrite the canonical phone_numbers field with E.164-normalized values."""
    result = []
    for items in phones:
        lst = list(items) if items is not None else []
        normalized = _normalize_phone_list(lst)
        result.append(json.dumps(normalized) if normalized else None)
    return pd.Series(result)


print("Phone normalization functions defined.")

# COMMAND ----------
# MAGIC %md ## 4. Apply JSON Parsers + Phone Normalization

# COMMAND ----------

ARRAY_COLUMNS = [
    "specialties", "procedure", "equipment", "capability",
    "phone_numbers", "affiliationtypeids", "countries", "websites",
]

df = bronze
for col in ARRAY_COLUMNS:
    df = df.withColumn(f"{col}_parsed", safe_json_udf(F.col(col)))

df = df.withColumn(
    "phone_numbers_parsed",
    normalize_phone_list_udf(F.col("phone_numbers_parsed"))
)

print("Array parsing + phone normalization complete.")
df.select("name", "specialties_parsed", "capability_parsed", "phone_numbers_parsed").show(5, truncate=80)

# COMMAND ----------
# MAGIC %md ## 5. Rename Bronze Columns + Country Normalization + Canonical Phone Rewrite

# COMMAND ----------

BRONZE_TO_SILVER_RENAMES = {
    "officialwebsite":       "officialWebsite",
    "address_stateorregion": "address_stateOrRegion",
    "address_ziporpostcode": "address_zipOrPostcode",
    "address_countrycode":   "address_countryCode",
    "facilitytypeid":        "facilityTypeId",
    "operatortypeid":        "operatorTypeId",
}

for old, new in BRONZE_TO_SILVER_RENAMES.items():
    df = df.withColumnRenamed(old, new)

df = df \
    .withColumn(
        "address_country",
        F.when(F.lower(F.trim(F.col("address_country"))).isin("gh", "ghana"), F.lit("Ghana"))
         .otherwise(F.col("address_country"))
    ) \
    .withColumn(
        "address_countryCode",
        F.when(
            F.col("address_countryCode").isNull() & (F.col("address_country") == "Ghana"),
            F.lit("GH")
        ).when(F.col("address_countryCode").isNotNull(), F.upper(F.col("address_countryCode")))
         .otherwise(F.col("address_countryCode"))
    )

# v8-5: Rewrite canonical phone_numbers STRING field to E.164 normalized JSON
df = df.withColumn(
    "phone_numbers",
    normalized_phone_numbers_string_udf(F.col("phone_numbers_parsed"))
)

print("Renamed + normalized country + rewrote phone_numbers to E.164.")

# COMMAND ----------
# MAGIC %md ## 6. Specialty Vocabulary Normalization (v8 — FDR canonical camelCase)

# COMMAND ----------

# Fuzzy-match map: common misspellings / alternate capitalizations → canonical
_SPECIALTY_ALT_MAP: Dict[str, str] = {
    # Common alternate forms
    "internalmedicine":                          "internalMedicine",
    "familymedicine":                            "familyMedicine",
    "generalsurgery":                            "generalSurgery",
    "emergencymedicine":                         "emergencyMedicine",
    "gynecologyandobstetrics":                   "gynecologyAndObstetrics",
    "gynaecologyandobstetrics":                  "gynecologyAndObstetrics",
    "orthopedicsurgery":                         "orthopedicSurgery",
    "orthopaedicsurgery":                        "orthopedicSurgery",
    "infectiousdiseases":                        "infectiousDiseases",
    "criticalcaremedicine":                      "criticalCareMedicine",
    "cardiacsurgery":                            "cardiacSurgery",
    "plasticsurgery":                            "plasticSurgery",
    "endocrinologyanddiabetesandmetabolism":     "endocrinologyAndDiabetesAndMetabolism",
    "neonatologyperinatalmedicine":              "neonatologyPerinatalMedicine",
    "medicaloncology":                           "medicalOncology",
    "physicalmedicineandrehabilitation":         "physicalMedicineAndRehabilitation",
    "geriatricsinternal medicine":               "geriatricsInternalMedicine",
    "geriatricsinternal":                        "geriatricsInternalMedicine",
    "hospiceandpalliativeinternal medicine":     "hospiceAndPalliativeInternalMedicine",
    "publichealthandglobal":                     "publicHealth",
    "globalhealthandinternationalhealth":        "globalHealthAndInternationalHealth",
    "clinicalpathology":                         "clinicalPathology",
    "obstetricsandmaternity care":               "obstetricsAndMaternityCare",
    "obstetricsandmaternitycareservice":         "obstetricsAndMaternityCare",
    "reproductiveendocrinologyandfertility":     "reproductiveEndocrinologyAndInfertility",
    "maternalfetal medicine":                    "maternalFetalMedicineOrPerinatology",
    "maternalfetalmedicineorperinatology":       "maternalFetalMedicineOrPerinatology",
    "socialandbehavioralsciences":               "socialAndBehavioralSciences",
    "familyplanningandcomplexcontraception":     "familyPlanningAndComplexContraception",
}


def _normalize_specialty(s: str) -> Optional[str]:
    """Normalize a single specialty string to FDR canonical camelCase."""
    if not s:
        return None
    s = str(s).strip()
    # Already valid
    if s in FDR_SPECIALTY_VOCAB:
        return s
    # Try lowercase key lookup
    key = re.sub(r"[\s_\-]", "", s.lower())
    if key in _SPECIALTY_ALT_MAP:
        return _SPECIALTY_ALT_MAP[key]
    # Try removing spaces in canonical set
    for canonical in FDR_SPECIALTY_VOCAB:
        if canonical.lower() == key:
            return canonical
    return None  # Drop non-canonical values


@pandas_udf(ArrayType(StringType()))
def normalize_specialties_udf(specs: pd.Series) -> pd.Series:
    result = []
    for items in specs:
        lst = list(items) if items is not None else []
        normalized = []
        seen = set()
        for item in lst:
            canon = _normalize_specialty(str(item))
            if canon and canon not in seen:
                normalized.append(canon)
                seen.add(canon)
        result.append(normalized)
    return pd.Series(result)


df = df.withColumn("specialties_parsed", normalize_specialties_udf(F.col("specialties_parsed")))
print("Specialty normalization (FDR canonical camelCase) applied.")

# COMMAND ----------
# MAGIC %md ## 7. countries_parsed — ISO alpha-2 Validation + Ghana Enforcement

# COMMAND ----------

# Valid ISO alpha-2 codes (subset — expands as needed)
_VALID_ISO2: frozenset = frozenset({
    "GH", "NG", "CI", "SN", "ML", "BF", "TG", "BJ", "NE", "GN", "LR", "SL",
    "GM", "GW", "CM", "GA", "CG", "CD", "AO", "ZA", "KE", "TZ", "UG", "RW",
    "ET", "SD", "SO", "MZ", "ZW", "ZM", "MW", "MG", "MR", "MA", "TN", "EG",
    "LY", "DZ", "GB", "US", "DE", "FR", "IT", "NL", "SE", "NO", "DK", "CH",
    "CA", "AU", "IN", "CN", "JP", "BR", "AR", "MX",
})

_COUNTRY_TO_ISO2: Dict[str, str] = {
    "ghana": "GH", "nigeria": "NG", "ivory coast": "CI", "cote d'ivoire": "CI",
    "senegal": "SN", "mali": "ML", "burkina faso": "BF", "togo": "TG",
    "benin": "BJ", "niger": "NE", "guinea": "GN", "liberia": "LR",
    "sierra leone": "SL", "gambia": "GM", "guinea-bissau": "GW",
    "cameroon": "CM", "united kingdom": "GB", "uk": "GB",
    "united states": "US", "usa": "US", "germany": "DE", "france": "FR",
}


def _validate_countries(items: Optional[List], address_country: Optional[str]) -> List[str]:
    """Validate and normalize countries list to ISO alpha-2. Enforce GH for Ghana records."""
    valid = []
    seen = set()
    for item in (items or []):
        s = str(item).strip().upper()
        if s in _VALID_ISO2 and s not in seen:
            valid.append(s)
            seen.add(s)
            continue
        # Try lowercase lookup
        lower = str(item).strip().lower()
        iso = _COUNTRY_TO_ISO2.get(lower)
        if iso and iso not in seen:
            valid.append(iso)
            seen.add(iso)
    # Enforce GH when address_country is Ghana
    if address_country and str(address_country).strip().lower() in ("ghana", "gh"):
        if "GH" not in seen:
            valid.insert(0, "GH")
    return valid


@pandas_udf(ArrayType(StringType()))
def validate_countries_udf(countries: pd.Series, addr_country: pd.Series) -> pd.Series:
    result = []
    for items, ac in zip(countries, addr_country):
        lst = list(items) if items is not None else []
        result.append(_validate_countries(lst, ac))
    return pd.Series(result)


df = df.withColumn(
    "countries_parsed",
    validate_countries_udf(F.col("countries_parsed"), F.col("address_country"))
)
print("Countries ISO alpha-2 validation + Ghana enforcement applied.")

# COMMAND ----------
# MAGIC %md ## 8. organizationdescription Neutrality Pass (v8)

# COMMAND ----------

# Patterns flagging subjective/religious/promotional language
_SUBJECTIVE_RE = re.compile(
    r"(?i)\b("
    r"blessed|glory|god(?:'s)?|lord|jesus|christ|holy|divine|sacred|prayer|mission\s+of\s+god"
    r"|love\s+of\s+(?:god|jesus)|grace\s+of|faith-based|spirit(?:ual)?"
    r"|dedicated\s+to\s+serving\s+(?:god|the\s+lord)"
    r"|amazing|outstanding|excellent|world-class|best\s+in|unmatched|premier|leading"
    r"|passionate\s+about|committed\s+to\s+(?:excellence|quality)"
    r"|we\s+are\s+proud|proud\s+to\s+serve"
    r")\b"
)

# Sentence splitter
_SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def _neutralize_org_description(text: Optional[str]) -> Optional[str]:
    """
    Strip subjective, religious, and promotional language from organizationdescription.
    Keeps factual sentences only. Returns None if nothing survives.
    """
    if not text or str(text).strip() in ("", "null", "None", "nan"):
        return None
    text = str(text).strip()
    sentences = _SENT_SPLIT_RE.split(text)
    clean = []
    for sent in sentences:
        sent = sent.strip()
        if len(sent) < 10:
            continue
        if _SUBJECTIVE_RE.search(sent):
            continue
        clean.append(sent)
    result = " ".join(clean).strip()
    return result if len(result) > 15 else None


@pandas_udf(StringType())
def neutralize_org_description_udf(desc: pd.Series) -> pd.Series:
    return desc.apply(_neutralize_org_description)


df = df.withColumn(
    "organizationdescription",
    neutralize_org_description_udf(F.col("organizationdescription"))
)
print("organizationdescription neutrality pass applied.")

# COMMAND ----------
# MAGIC %md ## 9. Junk Filter — v8: Capability Whitelist for High-Signal Phrases

# COMMAND ----------

# High-signal clinical phrases that MUST NEVER be filtered
# regardless of other junk patterns (whitelist takes priority)
_CAPABILITY_WHITELIST_RE = re.compile(
    r"(?i)\b("
    r"ICU|NICU|HDU|CCU"
    r"|intensive\s+care\s+unit"
    r"|high\s+dependency\s+unit"
    r"|trauma\s+(centre|center|level|care)"
    r"|24[\s\-]?hour\s+(emergency|obstetric|casualty|care|service)"
    r"|emergency\s+(department|unit|service|care|obstetric)"
    r"|operating\s+(theatre|room|theater)"
    r"|inpatient\s+(ward|service|bed)"
    r"|outpatient\s+(department|service|clinic)"
    r"|dialysis\s+(unit|service|centre)"
    r"|blood\s+bank"
    r"|mortuary"
    r"|PMTCT"
    r"|antiretroviral"
    r"|oxygen\s+(plant|concentrator|supply)"
    r"|ambulance\s+(service|bay)"
    r"|maternity\s+(ward|unit|block)"
    r"|neonatal\s+(unit|ward|care)"
    r")\b"
)

_JUNK_TOKEN_RE = re.compile(
    r"("
    r"^located\s+(at|in)\b"
    r"|^has\s+a\s+location\s+at\b"
    r"|^p\.?\s*o\.?\s*box\b"
    r"|GPS\s+address"
    r"|\b(?:suite|floor|building)\s+\w+\b"
    r"|telephone\s+numbers?\s+(?:are|is)"
    r"|has\s+(?:a\s+)?(?:telephone|phone)\s+number"
    r"|has\s+(?:an?\s+)?email\s+address"
    r"|^phone\s*(?:number|contact)?\s*:"
    r"|^contact\s+(?:number|info|us)?\s*:"
    r"|^fax\s*:"
    r"|^email\s*:"
    r"|\btel(?:ephone)?\s*[:\-]"
    r"|\+\d{7,}"
    r"|http[s]?://"
    r"|www\."
    r"|facebook|instagram|twitter|whatsapp|linkedin"
    r"|official\s+website"
    r"|\bis\s+(?:privately|government|publicly)\s+owned\b"
    r"|\bOwnership\s*[:\-]"
    r"|\bType\s*[:\-]\s*(?:Primary|Secondary|Tertiary|District|Regional|Community|CHPS|Health)"
    r"|\bFacility\s+type\s*[:\-]"
    r"|\bNHIS\s+(?:accredited|accreditation|registered)\b"
    r"|\bprovides\s+general\s+(?:medical\s+|health\s+)?services?\b"
    r"|\bServices\s*[:\-]\s*General\b"
    r"|listed\s+(as|in)\s+"
    r"|registered\s+with\s+Ghana"
    r"|page\s+created\s+on"
    r"|\d+\s+(?:people\s+)?(?:like|follow|check.?in)"
    r"|opening\s+hours?"
    r"|business\s+hours?"
    r"|\bMon\s*[-–]\s*(?:Fri|Sun)\b"
    r"|\b\d{1,2}:\d{2}\s*(?:am|pm|AM|PM)\b"
    r"|\blikes?\b"
    r"|\bfollowers?\b"
    r"|\bcheck.?ins?\b"
    r"|\brating\b"
    r"|\breviews?\b"
    r"|\bstars?\b"
    r")",
    re.IGNORECASE,
)

_JUNK_SENTENCE_RE = re.compile(
    r"("
    r"(?:the\s+facility\s+)?(?:is\s+)?(?:located|situated|found)\s+(?:at|in|on)\s+.{10,}"
    r"|(?:our\s+)?(?:address|location)\s+is\s+.{5,}"
    r"|(?:you\s+can\s+)?(?:find|reach|contact)\s+us\s+(?:at|on)\s+.{5,}"
    r"|(?:call|contact|reach)\s+us\s+(?:on|at)\s+[\d\+\s\-\(\)]{7,}"
    r"|(?:phone|tel(?:ephone)?)\s*[:\-]\s*[\+\d][\d\s\-\(\)]{6,}"
    r"|for\s+(?:more\s+)?(?:information|enquiries?|appointments?),?\s+(?:call|contact|visit)"
    r"|(?:this\s+)?(?:facility|hospital|clinic|centre)\s+(?:is\s+)?(?:owned|operated|managed|run)\s+by\s+"
    r")",
    re.IGNORECASE,
)

_CLINICAL_INDICATORS = re.compile(
    r"\b("
    r"treat(?:ment|s|ing)?|diagnos(?:e|is|tic)?|therap(?:y|ies|ist)?"
    r"|surger(?:y|ies)|surgical|clinic(?:al)?|hospital|ward|patient"
    r"|doctor|physician|nurse|specialist|consult(?:ation)?"
    r"|procedure|service|care|health|medical|medicine|prescription"
    r"|laboratory|lab|delivery|antenatal|postnatal|obstetric|maternity"
    r"|pediatric|geriatric|dental|eye|optic|optometr|vision|ophthalmolog"
    r"|x.?ray|ultrasound|scan|ecg|mri|ct\s+scan|emergency|resuscit|trauma"
    r"|first\s+aid|vaccination|immunis|pharmacy|dispensar"
    r")\b",
    re.IGNORECASE,
)


def _apply_junk_filter(items: Optional[List[str]]) -> List[str]:
    """v8: Whitelist protects high-signal clinical phrases before junk filtering."""
    if not items:
        return []
    seen = set()
    out  = []
    for item in items:
        if not item:
            continue
        s = str(item).strip()
        if len(s) < 8 or len(s) > 500:
            continue

        # v8: Whitelist check — never filter high-signal clinical phrases
        if _CAPABILITY_WHITELIST_RE.search(s):
            key = re.sub(r"[^\w\s]", "", s.lower())
            key = re.sub(r"\s+", " ", key).strip()
            if key not in seen:
                seen.add(key)
                out.append(s)
            continue

        if _JUNK_TOKEN_RE.search(s):
            continue
        if _JUNK_SENTENCE_RE.search(s):
            continue
        if re.fullmatch(r"[\d\s\.,\+\-\(\)\/]+", s):
            continue
        if len(s) > 50 and not _CLINICAL_INDICATORS.search(s):
            continue
        key = re.sub(r"[^\w\s]", "", s.lower())
        key = re.sub(r"\s+", " ", key).strip()
        if key not in seen:
            seen.add(key)
            out.append(s)
    return out


junk_filter_udf = F.udf(_apply_junk_filter, ArrayType(StringType()))

for col in ["capability_parsed", "procedure_parsed", "equipment_parsed"]:
    df = df.withColumn(col, junk_filter_udf(F.col(col)))

print("Junk filter (v8: whitelist + sentence-level + clinical gate) applied.")

# COMMAND ----------
# MAGIC %md ## 10. Mine Procedures & Equipment (v7 vocab + v8 context-inference)

# COMMAND ----------

PROCEDURE_KEYWORDS = [
    "general surgery", "surgical", "caesarean section", "c-section", "caesarean",
    "laparoscopy", "laparoscopic", "appendectomy", "hysterectomy", "herniorrhaphy",
    "hernia repair", "cholecystectomy", "thyroidectomy", "mastectomy",
    "cataract surgery", "laser surgery", "eye surgery", "corneal transplant",
    "hip replacement", "knee replacement", "orthopaedic surgery", "orthopedic surgery",
    "prostatectomy", "nephrectomy", "colostomy", "bowel resection",
    "antenatal care", "postnatal care", "delivery", "obstetrics",
    "gynecology", "gynaecology", "family planning", "fertility treatment",
    "maternity services", "labour and delivery", "normal delivery",
    "laboratory", "lab tests", "blood test", "urine test", "stool test",
    "medical lab", "in-house lab", "laboratory services",
    "ultrasound", "ultrasound scan", "x-ray", "x ray", "x-rays", "radiography",
    "digital x-ray", "ct scan", "mri", "mammography",
    "ecg", "electrocardiogram", "echocardiogram",
    "endoscopy", "colonoscopy", "biopsy", "pap smear",
    "audiometry", "spirometry", "pulmonary function test",
    "blood transfusion", "hemodialysis", "dialysis", "chemotherapy",
    "radiotherapy", "physiotherapy", "physical therapy", "occupational therapy",
    "speech therapy", "wound care", "wound dressing", "suturing",
    "vaccination", "immunisation", "immunization", "circumcision",
    "minor surgery", "minor procedures", "outpatient procedures",
    "electro convulsive therapy", "ect", "psychotherapy", "counselling", "counseling",
    "psychiatric services", "mental health services",
    "dental extraction", "tooth extraction", "dental filling", "root canal",
    "dental cleaning", "scaling and polishing", "orthodontic", "dentures",
    "dental services", "in-house dental lab",
    "eye examination", "vision testing", "refraction", "glaucoma treatment",
    "ophthalmology services", "optometry services", "eye care", "eye care services",
    "cataract screening", "low vision",
    "emergency care", "resuscitation", "trauma care", "first aid", "emergency services",
    "dispensary", "pharmacy services", "pharmaceutical services",
    "paediatric care", "pediatric care", "neonatal care", "newborn care",
    "child health", "child welfare",
    "dermatology", "neurology", "cardiology", "endocrinology", "urology",
    "gastroenterology", "haematology", "hematology", "oncology", "geriatrics",
]

EQUIPMENT_KEYWORDS = [
    "x-ray machine", "x ray machine", "x-rays", "digital x-ray", "portable x-ray",
    "ultrasound machine", "ultrasound scan", "ultrasound scanner",
    "ct scanner", "ct scan", "mri machine", "mri scanner",
    "mammography machine", "fluoroscopy",
    "doppler", "doppler ultrasound", "echocardiograph", "echocardiography",
    "haematology analyser", "hematology analyzer", "biochemistry analyser",
    "blood gas analyser", "pcr machine", "centrifuge", "microscope",
    "cd4 machine", "viral load machine",
    "medical laboratory", "medical lab", "in-house laboratory", "in-house lab",
    "laboratory equipment",
    "operating theatre", "operating room", "operating theater",
    "anaesthesia machine", "anesthesia machine",
    "ventilator", "mechanical ventilation", "defibrillator",
    "laparoscope", "endoscope", "operating table",
    "ecg machine", "ecg", "electrocardiogram machine",
    "pulse oximeter", "vital signs monitor",
    "cardiac monitor", "blood pressure monitor", "glucometer",
    "infusion pump", "syringe pump",
    "icu bed", "icu unit", "intensive care unit", "high dependency unit",
    "oxygen plant", "oxygen concentrator", "oxygen cylinder",
    "ambulance", "blood bank", "mortuary",
    "dialysis machine", "haemodialysis", "hemodialysis machine",
    "dental chair", "dental unit", "dental x-ray", "in-house dental lab",
    "slit lamp", "tonometer", "phoropter", "fundus camera",
    "automated dispensing", "cold chain",
]

# v8: Context-based equipment inference rules
# Format: (trigger_regex, equipment_items_to_add)
EQUIPMENT_CONTEXT_RULES: List[Tuple[re.Pattern, List[str]]] = [
    (re.compile(r"\b(laboratory\s+services?|medical\s+lab|in-house\s+lab|lab\s+tests?)\b", re.I),
     ["Laboratory Equipment", "Microscope", "Centrifuge"]),
    (re.compile(r"\b(imaging|radiology\s+services?|scan\s+cent)\b", re.I),
     ["X-Ray Machine", "Ultrasound Machine"]),
    (re.compile(r"\b(icu|intensive\s+care\s+unit)\b", re.I),
     ["Ventilator", "Cardiac Monitor", "Pulse Oximeter", "Infusion Pump"]),
    (re.compile(r"\b(emergency\s+(department|unit|services?)|casualty\s+unit)\b", re.I),
     ["Defibrillator", "Vital Signs Monitor", "Oxygen Cylinder"]),
    (re.compile(r"\b(maternity\s+(ward|unit)|obstetric\s+care|labour\s+ward)\b", re.I),
     ["Fetal Monitor", "Delivery Bed", "Oxygen Concentrator"]),
    (re.compile(r"\b(eye\s+clinic|ophthalmolog|optometr)\b", re.I),
     ["Slit Lamp", "Tonometer", "Fundus Camera"]),
    (re.compile(r"\b(dialysis|hemodialysis|haemodialysis)\b", re.I),
     ["Hemodialysis Machine"]),
    (re.compile(r"\b(operating\s+(theatre|room)|surgical\s+(suite|unit))\b", re.I),
     ["Operating Theatre", "Anaesthesia Machine", "Operating Table"]),
    (re.compile(r"\b(blood\s+bank|transfusion\s+services?)\b", re.I),
     ["Blood Bank", "Centrifuge"]),
    (re.compile(r"\b(oxygen\s+(plant|concentrator|supply|therapy))\b", re.I),
     ["Oxygen Plant"]),
]


def _mine_from_description(desc: Optional[str], vocab: List[str]) -> List[str]:
    if not desc or str(desc).strip() in ("", "null", "None", "nan"):
        return []
    text = str(desc).lower()
    found, seen = [], set()
    for term in vocab:
        t = term.lower()
        pattern = r'\b' + re.escape(t) + r'\b'
        if re.search(pattern, text):
            if t not in seen:
                seen.add(t)
                found.append(term.title())
    return found


@pandas_udf(ArrayType(StringType()))
def mine_procedures_udf(
    proc_arr: pd.Series, desc_col: pd.Series, cap_arr: pd.Series,
) -> pd.Series:
    result = []
    for procs, desc, caps in zip(proc_arr, desc_col, cap_arr):
        combined = list(procs) if isinstance(procs, (list, tuple)) else ([] if procs is None else list(procs))
        seen_keys = {re.sub(r"[^\w\s]", "", p.lower()).strip() for p in combined}
        for m in _mine_from_description(desc, PROCEDURE_KEYWORDS):
            k = re.sub(r"[^\w\s]", "", m.lower()).strip()
            if k not in seen_keys:
                combined.append(m); seen_keys.add(k)
        caps_iter = caps if isinstance(caps, (list, tuple)) else ([] if caps is None else list(caps))
        for cap_item in caps_iter:
            if not cap_item or len(cap_item) < 10:
                continue
            c_text = str(cap_item).lower()
            for kw in PROCEDURE_KEYWORDS:
                if re.search(r'\b' + re.escape(kw.lower()) + r'\b', c_text):
                    k = kw.lower()
                    if k not in seen_keys:
                        combined.append(kw.title()); seen_keys.add(k)
        result.append(combined)
    return pd.Series(result)


@pandas_udf(ArrayType(StringType()))
def mine_equipment_udf(
    equip_arr: pd.Series, desc_col: pd.Series, cap_arr: pd.Series,
) -> pd.Series:
    result = []
    for equips, desc, caps in zip(equip_arr, desc_col, cap_arr):
        combined = list(equips) if equips is not None else []
        seen_keys = {re.sub(r"[^\w\s]", "", e.lower()).strip() for e in combined}

        # Keyword mining from description
        for m in _mine_from_description(desc, EQUIPMENT_KEYWORDS):
            k = re.sub(r"[^\w\s]", "", m.lower()).strip()
            if k not in seen_keys:
                combined.append(m); seen_keys.add(k)

        # Keyword mining from capability
        caps_iter = caps if isinstance(caps, (list, tuple)) else ([] if caps is None else list(caps))
        cap_text_full = " ".join(str(c) for c in caps_iter)
        for cap_item in caps_iter:
            if not cap_item or len(cap_item) < 10:
                continue
            c_text = str(cap_item).lower()
            for kw in EQUIPMENT_KEYWORDS:
                if re.search(r'\b' + re.escape(kw.lower()) + r'\b', c_text):
                    k = kw.lower()
                    if k not in seen_keys:
                        combined.append(kw.title()); seen_keys.add(k)

        # v8: Context-based inference from capability + description combined
        combined_text = f"{str(desc or '')} {cap_text_full}"
        for pattern, items_to_add in EQUIPMENT_CONTEXT_RULES:
            if pattern.search(combined_text):
                for item in items_to_add:
                    k = re.sub(r"[^\w\s]", "", item.lower()).strip()
                    if k not in seen_keys:
                        combined.append(item); seen_keys.add(k)

        result.append(combined)
    return pd.Series(result)


df = df \
    .withColumn("procedure_parsed",
                mine_procedures_udf(F.col("procedure_parsed"), F.col("description"), F.col("capability_parsed"))) \
    .withColumn("equipment_parsed",
                mine_equipment_udf(F.col("equipment_parsed"), F.col("description"), F.col("capability_parsed")))

print("Procedure/equipment mining (v8: context-inference) complete.")
df.select("name", "procedure_parsed", "equipment_parsed").show(8, truncate=100)

# COMMAND ----------
# MAGIC %md ## 11. Numeric Field Extraction — v7 patterns + v8 rule-based second layer

# COMMAND ----------

_TIME_SOCIAL_GUARD = re.compile(
    r"(?i)("
    r"24\s*/\s*7|24\s*hours?|\bhours?\b|\bdays?\s+a\s+week\b"
    r"|\bopen\s+\d|\blikes?\b|\bfollowers?\b|\bcheck.?ins?\b"
    r"|\brating\b|\breviews?\b|\bstars?\b"
    r"|\bsince\s+\d{4}\b|\bin\s+\d{4}\b|\bestablished\s+\d{4}\b"
    r"|\bfounded\s+\d{4}\b|\bopened\s+\d{4}\b|\bstarted\s+\d{4}\b"
    r")"
)

_DOC_PATTERNS = [
    re.compile(r"medical\s+team\s+(?:consists?\s+of|of)\s+(\d+)", re.I),
    re.compile(r"team\s+of\s+(\d+)\s+(?:french\s+and\s+ghanaian\s+)?(?:medical|health(?:care)?|clinical)\s+(?:professionals?|staff|workers?|practitioners?|clinicians?)", re.I),
    re.compile(r"team\s+of\s+(\d+)\s+(?:doctors?|physicians?|specialists?|surgeons?|general\s+practitioners?|clinicians?)", re.I),
    re.compile(r"staff\s+of\s+(\d+)\s+(?:doctors?|physicians?|clinicians?|medical\s+professionals?)", re.I),
    re.compile(r"(?:employs?|has)\s+(\d+)\s+(?:medical\s+)?(?:doctors?|physicians?|specialists?|clinicians?)", re.I),
    re.compile(r"staffed\s+(?:by|with)\s+(\d+)\s+(?:medical\s+)?(?:professionals?|doctors?|physicians?)", re.I),
    re.compile(r"(\d+)\s+(?:permanent|full.time|part.time|resident)\s+(?:medical\s+)?(?:doctors?|physicians?|staff)", re.I),
    re.compile(r"(\d+)\s+(?:medical\s+)?(?:doctors?|physicians?|specialists?|consultants?|surgeons?|clinicians?)\s+(?:are|on\s+call|available|on\s+duty|serving)", re.I),
    re.compile(r"(\d+)\s+(?:french\s+and\s+ghanaian\s+)?medical\s+professionals?", re.I),
    re.compile(r"(\d+)\s+(?:doctors?|physicians?|specialists?|consultants?)\s+(?:and|&)\s+\d+\s+(?:nurses?|staff)", re.I),
    re.compile(r"\b(\d{1,3})\s+(?:medical\s+)?(?:doctors?|physicians?|specialists?|consultants?|surgeons?|general\s+practitioners?)\b", re.I),
    re.compile(r"\b(\d{1,3})\s+(?:staff|health\s+workers|medical\s+staff)\b", re.I),
    re.compile(r"\bteam\s+size\s+(?:of\s+)?(\d{1,3})\b", re.I),
]
_DOC_PATTERNS.extend([
    re.compile(r"\bteam\s+of\s+(\d{1,3})\b", re.I),
    re.compile(r"\bstaff\s+strength\s+(?:of\s+)?(\d{1,3})\b", re.I),
    re.compile(r"\btotal\s+staff\s+(?:of\s+)?(\d{1,3})\b", re.I),
    re.compile(r"\bworkforce\s+(?:of\s+)?(\d{1,3})\b", re.I),
])
_DOC_PATTERNS.extend([
    re.compile(r"\b(\d{1,3})\s+doctors?\s+(?:and|&)\s+\d+\s+(?:nurses?|staff)\b", re.I),
    re.compile(r"\b(\d{1,3})\s+physicians?\s+(?:and|&)\s+\d+\s+(?:nurses?|staff)\b", re.I),
    re.compile(r"\b(\d{1,3})\s+consultants?\s+(?:and|&)\s+\d+\s+(?:support\s+staff)\b", re.I),
])
_DOC_PATTERNS.extend([
    re.compile(r"\b(\d{1,3})\s+doctors?\s+(?:available|present|on\s+site)\b", re.I),
    re.compile(r"\b(\d{1,3})\s+doctors?\s+(?:serving|serves|serve)\b", re.I),
    re.compile(r"\b(\d{1,3})\s+doctors?\s+(?:attached|affiliated)\b", re.I),
])
_DOC_PATTERNS.extend([
    re.compile(r"\b(\d{1,3})\s+consultants?\s+across\s+departments\b", re.I),
    re.compile(r"\b(\d{1,3})\s+specialists?\s+across\b", re.I),
    re.compile(r"\b(\d{1,3})\s+medical\s+officers?\b", re.I),
])
_DOC_PATTERNS.extend([
    re.compile(r"\b(\d{1,3})\s+doctors?\s+per\s+shift\b", re.I),
    re.compile(r"\b(\d{1,3})\s+doctors?\s+on\s+duty\b", re.I),
])
_DOC_PATTERNS.extend([
    re.compile(r"\b(\d{1,3})\s+specialists?\b", re.I),
    re.compile(r"\b(\d{1,3})\s+consultants?\b", re.I),
    re.compile(r"\b(\d{1,3})\s+surgeons?\b", re.I),
])

_MONTHS_PAT = r"(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)"
_BED_PATTERNS = [
    re.compile(r"\b(\d{2,4})\s*[-\u2013\u2014]\s*\d{2,4}\s*beds?\b", re.I),
    re.compile(r"\b(\d{1,4})[+]?\s*[-]?\s*bed\s+(?:hospital|facility|centre|center|capacity)\b", re.I),
    re.compile(r"\b(\d{1,4})-bed\b", re.I),
    re.compile(r"\b(\d{2,4})\s+bed\s+capacity\b", re.I),
    re.compile(r"bed\s+capacity\s*(?:of|:)?\s*(\d{2,4})", re.I),
    re.compile(r"bedspace\s+(?:of|:)?\s*(\d{2,4})", re.I),
    re.compile(r"\b(\d{2,4})\s+bedspaces?\b", re.I),
    re.compile(r"capacity\s+(?:of|to\s+accommodate)\s+(\d{2,4})\s*(?:patients?|beds?)?", re.I),
    re.compile(r"accommodate\s+(\d{2,4})\s+(?:beds?|patients?|inpatients?)", re.I),
    re.compile(r"(?:currently\s+)?running\s+(\d{2,4})\s*beds?", re.I),
    re.compile(r"operating\s+(?:a\s+)?(\d{2,4})\s*[-]?\s*bed", re.I),
    re.compile(r"\b(\d{2,4})\s+beds?\s+(?:plus|and)\b", re.I),
    re.compile(r"(?:is\s+a|a|an)\s+(\d{2,4})\s*[-]?\s*bed\b", re.I),
    re.compile(r"\b(\d{2,4})\s+(?:beds?|inpatient|ward\s+beds?)\b", re.I),
]
_BED_PATTERNS.extend([
    re.compile(r"\binpatient\s+capacity\s*(?:of|:)?\s*(\d{2,4})\b", re.I),
    re.compile(r"\b(\d{2,4})\s+inpatient\s+beds?\b", re.I),
    re.compile(r"\b(\d{2,4})\s+ward\s+beds?\b", re.I),
    re.compile(r"\b(\d{2,4})\s+wards?\s+with\s+\d+\s+beds?\b", re.I),
])
_BED_PATTERNS.extend([
    re.compile(r"\bhas\s+(\d{2,4})\s+beds?\b", re.I),
    re.compile(r"\bcurrently\s+has\s+(\d{2,4})\s+beds?\b", re.I),
    re.compile(r"\bprovides\s+(\d{2,4})\s+beds?\b", re.I),
    re.compile(r"\boffers\s+(\d{2,4})\s+beds?\b", re.I),
])
_BED_PATTERNS.extend([
    re.compile(r"\b(\d{2,4})\s*bed\s+(?:district|regional|teaching)\s+hospital\b", re.I),
    re.compile(r"\b(\d{2,4})\s*bed\s+medical\s+center\b", re.I),
    re.compile(r"\b(\d{2,4})\s*bed\s+health\s+facility\b", re.I),
])
_BED_PATTERNS.extend([
    re.compile(r"\bexpand(?:ed|ing)?\s+to\s+(\d{2,4})\s+beds?\b", re.I),
    re.compile(r"\bplanned\s+capacity\s+(?:of\s+)?(\d{2,4})\s+beds?\b", re.I),
    re.compile(r"\bdesigned\s+for\s+(\d{2,4})\s+beds?\b", re.I),
])
_BED_PATTERNS.extend([
    re.compile(r"\b(\d{1,3})\s+icu\s+beds?\b", re.I),
    re.compile(r"\bnicu\s+(?:with\s+)?(\d{1,3})\s+beds?\b", re.I),
    re.compile(r"\b(\d{1,3})\s+intensive\s+care\s+beds?\b", re.I),
])
_BED_PATTERNS.extend([
    re.compile(r"\bcan\s+accommodate\s+up\s+to\s+(\d{2,4})\s+(?:patients?|beds?)\b", re.I),
    re.compile(r"\baccommodation\s+for\s+(\d{2,4})\s+(?:patients?|beds?)\b", re.I),
])
_YEAR_PATTERNS = [
    re.compile(rf"(?:founded|established|started|commissioned|opened)\s+(?:in\s+|on\s+)?(?:{_MONTHS_PAT}\s+)?(\d{{4}})\b", re.I),
    re.compile(r"since\s+(\d{4})\b", re.I),
    re.compile(r"serving\s+ghana\s+since\s+(\d{4})\b", re.I),
    re.compile(r"in\s+(\d{4})\s+(?:by|as\s+a)", re.I),
    re.compile(r"(?:maternity\s+home\s+in|health\s+centre\s+in|hospital\s+in)\s+(\d{4})\b", re.I),
    re.compile(r"\b(\d{4})\s+(?:to\s+date|to\s+present|till\s+date)", re.I),
    re.compile(r"(?:from|in)\s+(\d{4})\s+(?:to|till|until)\b", re.I),
    re.compile(r"\b(est\.?|established)\s*[:\-]?\s*(\d{4})\b", re.I),
    re.compile(r"\bfounded\s*[:\-]?\s*(\d{4})\b", re.I),
]

_AREA_PATTERNS = [
    re.compile(r"(\d[\d,]+)\s*(?:sq(?:uare)?\s*(?:met(?:ers?|res?)|m[²2]))", re.I),
    re.compile(r"floor\s+(?:area|space)\s*(?:of|:)?\s*(\d[\d,]+)", re.I),
    re.compile(r"(\d[\d,]+)\s*m²", re.I),
]

# v8: Rule-based bed capacity ranges by facility type (used when regex finds nothing)
FTYPE_BED_RANGE: Dict[str, Tuple[int, int]] = {
    "hospital":          (50, 200),
    "clinic":            (5, 30),
    "maternity_home":    (10, 40),
    "chps":              (2, 10),
    "eye_clinic":        (0, 5),
    "diagnostic_center": (0, 0),
    "pharmacy":          (0, 0),
    "dentist":           (0, 5),
    "doctor":            (0, 5),
}


def _plausibility_check(val: int, field: str) -> bool:
    if field == "doctors": return 1 <= val <= cfg.MAX_DOCTORS
    if field == "beds":    return 5 <= val <= cfg.MAX_BEDS
    if field == "year":    return cfg.MIN_YEAR <= val <= cfg.MAX_YEAR
    if field == "area":    return 10 <= val <= cfg.MAX_AREA
    return True


def _extract_int_from_text(
    text: str, patterns: List,
    cap: int = 9999, guard: Optional[re.Pattern] = None,
    min_val: int = 1, field: str = ""
) -> Optional[int]:
    if not text:
        return None
    for pat in patterns:
        for m in pat.finditer(text):
            if guard:
                ctx = text[max(0, m.start() - 60):m.end() + 60]
                if guard.search(ctx):
                    continue
            try:
                raw = str(m.group(1)).replace(",", "")
                val = int(raw)
                if min_val <= val <= cap and (not field or _plausibility_check(val, field)):
                    return val
            except (IndexError, ValueError):
                pass
    return None


def _extract_source_int(source_val, cap: int = 9999, min_val: int = 1) -> Optional[int]:
    if source_val is None:
        return None
    try:
        v = float(str(source_val).strip())
        if min_val <= v <= cap:
            return int(v)
    except (ValueError, TypeError):
        pass
    return None


@pandas_udf(ArrayType(StringType()))
def extract_number_doctors_udf(src:pd.Series, desc:pd.Series, org_desc:pd.Series, mission:pd.Series, cap:pd.Series) -> pd.Series:
    result = []
    for src_val, desc_val, org_desc_val, mission_val, caps in zip(
        src, desc, org_desc, mission, cap
    ):
        val = _extract_source_int(src_val, cap=cfg.MAX_DOCTORS)
        conf = "high" if val is not None else None
        if val is None:
            TEXT_SOURCES = [
                str(desc_val or ""),
                str(org_desc_val or ""),
                str(mission_val or ""),
            ]

            combined_text = " ".join(TEXT_SOURCES)
            val = _extract_int_from_text(combined_text, _DOC_PATTERNS,
                                         cap=cfg.MAX_DOCTORS, guard=_TIME_SOCIAL_GUARD, field="doctors")
            if val is not None: conf = "medium"
        if val is None and caps is not None:
            for cap_item in (list(caps) if isinstance(caps, (list, tuple)) else []):
                if cap_item and len(str(cap_item)) > 10:
                    v2 = _extract_int_from_text(str(cap_item), _DOC_PATTERNS,
                                                cap=cfg.MAX_DOCTORS, field="doctors")
                    if v2:
                        val = v2; conf = "low"; break
        result.append([str(val) if val is not None else "", conf or ""])
    return pd.Series(result)



@pandas_udf(ArrayType(StringType()))
def extract_capacity_int_udf(
    src: pd.Series,
    desc: pd.Series,
    org_desc: pd.Series,
    mission: pd.Series,
    cap: pd.Series,
    ftype: pd.Series
) -> pd.Series:
    result = []
    for src_val, desc_val, org_desc_val, mission_val, caps, ft in zip(
        src, desc, org_desc, mission, cap, ftype
    ):
        val = _extract_source_int(src_val, cap=cfg.MAX_BEDS, min_val=1)
        conf = "high" if val is not None else None
        if val is None:
            TEXT_SOURCES = [
                str(desc_val or ""),
                str(org_desc_val or ""),
                str(mission_val or ""),
            ]

            combined_text = " ".join(TEXT_SOURCES)
            val = _extract_int_from_text(combined_text, _BED_PATTERNS,
                                         cap=cfg.MAX_BEDS, min_val=5, field="beds")
            if val is not None: conf = "medium"
        if val is None and caps is not None:
            for cap_item in (list(caps) if isinstance(caps, (list, tuple)) else []):
                if cap_item and len(str(cap_item)) > 10:
                    v2 = _extract_int_from_text(str(cap_item), _BED_PATTERNS,
                                                cap=cfg.MAX_BEDS, min_val=5, field="beds")
                    if v2:
                        val = v2; conf = "low"; break
        # v8: Rule-based fallback from facility type (produces a range midpoint, low confidence)
        # STRICT inference: only if strong clinical signal exists
        if val is None and ft:
            ft_clean = str(ft).strip().lower()

            caps_list = list(caps) if isinstance(caps, (list, tuple)) else []
            text_blob = f"{str(desc_val or '')} {' '.join(str(c) for c in caps_list)}"

            STRONG_BED_SIGNAL = re.search(
                r"\b(inpatient|ward|admission|bed|hospitalization|icu|nicu|hdu)\b",
                text_blob,
                re.I
            )

            if STRONG_BED_SIGNAL and ft_clean in FTYPE_BED_RANGE:
                lo, hi = FTYPE_BED_RANGE[ft_clean]
                if hi > 0:
                    val = int((lo + hi) / 2)
                    conf = "inferred"   # <-- downgraded
        result.append([str(val) if val is not None else "", conf or ""])
    return pd.Series(result)


@pandas_udf(ArrayType(StringType()))
def extract_year_established_udf(
    src: pd.Series,
    desc: pd.Series,
    org_desc: pd.Series,
    mission: pd.Series
) -> pd.Series:
    result = []
    for src_val, desc_val, org_desc_val, mission_val in zip(
        src, desc, org_desc, mission
    ):
        val = _extract_source_int(src_val, cap=cfg.MAX_YEAR, min_val=cfg.MIN_YEAR)
        conf = "high" if val is not None else None
        if val is None:
            TEXT_SOURCES = [
                str(desc_val or ""),
                str(org_desc_val or ""),
                str(mission_val or ""),
            ]

            combined_text = " ".join(TEXT_SOURCES)
            val = _extract_int_from_text(combined_text, _YEAR_PATTERNS,
                                         cap=cfg.MAX_YEAR, min_val=cfg.MIN_YEAR, field="year")
            if val is not None: conf = "medium"
        result.append([str(val) if val is not None else "", conf or ""])
    return pd.Series(result)


@pandas_udf(IntegerType())
def extract_area_int_udf(src: pd.Series) -> pd.Series:
    result = []
    for src_val in src:
        result.append(_extract_source_int(src_val, cap=cfg.MAX_AREA, min_val=1))
    return pd.Series(result, dtype="Int64")


@pandas_udf(IntegerType())
def extract_pk_unique_id_int_udf(src: pd.Series) -> pd.Series:
    result = []
    for v in src:
        try:
            result.append(int(float(str(v).strip())))
        except:
            result.append(None)
    return pd.Series(result, dtype="Int64")


@pandas_udf(BooleanType())
def parse_accepts_volunteers_udf(src: pd.Series) -> pd.Series:
    def _parse(v):
        if v is None: return None
        s = str(v).strip().lower()
        if s in ("true", "1", "yes"):  return True
        if s in ("false", "0", "no"): return False
        return None
    return src.apply(_parse)


# Apply UDFs — store as arrays [value, confidence] then split
_doctors_arr = extract_number_doctors_udf(
    F.col("numberdoctors"),
    F.col("description"),
    F.col("organizationdescription"),
    F.col("missionstatement"),
    F.col("capability_parsed")
)
_capacity_arr = extract_capacity_int_udf(F.col("capacity"), F.col("description"),F.col("organizationdescription"),
    F.col("missionstatement"), F.col("capability_parsed"), F.col("facilityTypeId"))
_year_arr     = extract_year_established_udf(F.col("yearestablished"), F.col("description"),F.col("organizationdescription"),
    F.col("missionstatement"),)

df = df \
    .withColumn("_doctors_arr",  _doctors_arr) \
    .withColumn("_capacity_arr", _capacity_arr) \
    .withColumn("_year_arr",     _year_arr) \
    .withColumn("number_doctors_int",
                F.when(F.col("_doctors_arr").getItem(0) != "", F.col("_doctors_arr").getItem(0).cast(IntegerType())).otherwise(F.lit(None))) \
    .withColumn("doctors_confidence",
                F.when(F.col("_doctors_arr").getItem(1) != "", F.col("_doctors_arr").getItem(1)).otherwise(F.lit(None))) \
    .withColumn("capacity_int",
                F.when(F.col("_capacity_arr").getItem(0) != "", F.col("_capacity_arr").getItem(0).cast(IntegerType())).otherwise(F.lit(None))) \
    .withColumn("capacity_confidence",
                F.when(F.col("_capacity_arr").getItem(1) != "", F.col("_capacity_arr").getItem(1)).otherwise(F.lit(None))) \
    .withColumn("year_established_int",
                F.when(F.col("_year_arr").getItem(0) != "", F.col("_year_arr").getItem(0).cast(IntegerType())).otherwise(F.lit(None))) \
    .withColumn("year_confidence",
                F.when(F.col("_year_arr").getItem(1) != "", F.col("_year_arr").getItem(1)).otherwise(F.lit(None))) \
    .withColumn("area_int",                extract_area_int_udf(F.col("area"))) \
    .withColumn("accepts_volunteers_bool", parse_accepts_volunteers_udf(F.col("acceptsvolunteers"))) \
    .withColumn("pk_unique_id_int",        extract_pk_unique_id_int_udf(F.col("pk_unique_id"))) \
    .withColumn("numberDoctors",
                F.when(F.col("number_doctors_int").isNotNull(), F.col("number_doctors_int").cast(IntegerType()))
                 .otherwise(F.lit(None).cast(IntegerType()))) \
    .drop("_doctors_arr", "_capacity_arr", "_year_arr")

print("Numeric extraction (v8: with confidence levels) complete:")
for c in ["number_doctors_int", "capacity_int", "year_established_int", "area_int"]:
    cnt = df.filter(F.col(c).isNotNull()).count()
    print(f"  {c:<28} non-null: {cnt}")

print("\nConfidence breakdown for capacity_int:")
df.filter(F.col("capacity_int").isNotNull()).groupBy("capacity_confidence").count().show()

# COMMAND ----------
# MAGIC %md ## 12. Region Normalisation — v8: Kasoa hard override + No Ghana→Greater Accra fallback

# COMMAND ----------

VALID_REGIONS = frozenset({
    "Greater Accra", "Ashanti", "Western", "Eastern", "Central",
    "Volta", "Northern", "Upper East", "Upper West",
    "Oti", "Bono East", "Ahafo", "Savannah", "North East",
    "Western North", "Brong-Ahafo",
})

# v8 FIX: "Ghana" removed from REGION_NORM_MAP — country-only evidence must NOT
# produce a specific region. It stays Unknown.
REGION_NORM_MAP = {
    "Greater Accra Region": "Greater Accra", "Greater Accra": "Greater Accra",
    "Accra": "Greater Accra", "Accra Region": "Greater Accra",
    "Accra North": "Greater Accra", "Accra East": "Greater Accra",
    "East Legon": "Greater Accra", "North Legon": "Greater Accra",
    "Ga East Municipality": "Greater Accra",
    "Ga East Municipality, Greater Accra Region": "Greater Accra",
    "Shai Osudoku District, Greater Accra": "Greater Accra",
    "Ledzokuku-Krowor": "Greater Accra", "Tema West Municipal": "Greater Accra",
    "Ashanti Region": "Ashanti", "Ashanti": "Ashanti", "ASHANTI": "Ashanti",
    "Asante Region": "Ashanti", "Asante": "Ashanti",
    "Kumasi Metropolitan": "Ashanti", "Ejisu-Juaben Municipal": "Ashanti",
    "Ejisu Municipal": "Ashanti", "Ahafo Ano South-East": "Ashanti",
    "Ahafo Ano South East": "Ashanti", "SH": "Ashanti",
    "Western Region": "Western", "Western": "Western",
    "Eastern Region": "Eastern", "Eastern": "Eastern",
    "Central Region": "Central", "Central": "Central",
    "Central Ghana": "Central", "KEEA": "Central",
    "Cape Coast Municipal": "Central",
    "Awutu Senya East": "Central",
    "Awutu Senya East Municipal": "Central",
    "Volta Region": "Volta", "Volta": "Volta",
    "Central Tongu District": "Volta", "South Tongu": "Volta",
    "Northern Region": "Northern", "Northern": "Northern",
    "Tamale Metropolitan": "Northern",
    "Upper East Region": "Upper East", "Upper East": "Upper East",
    "Upper West Region": "Upper West", "Upper West": "Upper West",
    "Sissala West District": "Upper West", "Sissala East District": "Upper West",
    "Oti Region": "Oti", "Oti": "Oti",
    "Bono East Region": "Bono East", "Bono East": "Bono East",
    "Techiman Municipal": "Bono East",
    "Ahafo Region": "Ahafo", "Ahafo": "Ahafo", "Asutifi South": "Ahafo",
    "Savannah Region": "Savannah", "Savannah": "Savannah", "Damongo": "Savannah",
    "North East Region": "North East", "North East": "North East",
    "Western North Region": "Western North", "Western North": "Western North",
    "Brong Ahafo Region": "Brong-Ahafo", "Brong-Ahafo Region": "Brong-Ahafo",
    "Brong Ahafo": "Bono", "Brong-Ahafo": "Bono",
    "Bono Region": "Bono",
    "Bono": "Bono",
    "Sunyani": "Bono",
    # v8: "Ghana" deliberately NOT mapped to any region
}

CITY_TO_REGION: Dict[str, str] = {
    # ── Greater Accra ────────────────────────────────────────────────────────
    "accra": "Greater Accra", "tema": "Greater Accra", "ashaiman": "Greater Accra",
    "madina": "Greater Accra", "east legon": "Greater Accra", "north legon": "Greater Accra",
    "cantonments": "Greater Accra", "dansoman": "Greater Accra", "achimota": "Greater Accra",
    "lapaz": "Greater Accra", "spintex": "Greater Accra", "osu": "Greater Accra",
    "adenta": "Greater Accra", "adenta municipality": "Greater Accra",
    "teshie": "Greater Accra", "nungua": "Greater Accra", "adabraka": "Greater Accra",
    "jamestown": "Greater Accra", "labadi": "Greater Accra", "kokomlemle": "Greater Accra",
    "amasaman": "Greater Accra", "kwabenya": "Greater Accra", "dome": "Greater Accra",
    "oyarifa": "Greater Accra", "airport residential": "Greater Accra",
    "awoshie": "Greater Accra", "weija": "Greater Accra", "haatso": "Greater Accra",
    "east cantonments": "Greater Accra", "roman ridge": "Greater Accra",
    "kaneshie": "Greater Accra", "north kaneshie": "Greater Accra",
    "darkuman": "Greater Accra", "chorkor": "Greater Accra", "okponglo": "Greater Accra",
    "legon": "Greater Accra", "agbogba": "Greater Accra", "mempeasem": "Greater Accra",
    "ashale-botwe": "Greater Accra", "dzorwulu": "Greater Accra",
    "klagon": "Greater Accra", "odorkor": "Greater Accra", "pokoase": "Greater Accra",
    "pantang": "Greater Accra", "accra central": "Greater Accra",
    "agbogbloshie": "Greater Accra", "tesano": "Greater Accra", "labone": "Greater Accra",
    "ridge": "Greater Accra", "airport city": "Greater Accra",
    "accra newtown": "Greater Accra", "dodowa": "Greater Accra",
    "nsawam": "Eastern",
    "nsawam adoagyiri": "Eastern",
    # ── Ashanti ──────────────────────────────────────────────────────────────
    "kumasi": "Ashanti", "obuasi": "Ashanti", "ejisu": "Ashanti",
    "asokore": "Ashanti", "atonsu": "Ashanti", "atonsu kumasi": "Ashanti",
    "suame": "Ashanti", "bantama": "Ashanti", "nhyiaeso": "Ashanti",
    "asawase": "Ashanti", "tafo": "Ashanti", "kwadaso": "Ashanti",
    "manso nkwanta": "Ashanti", "juaben": "Ashanti", "mampong": "Ashanti",
    "nkawie": "Ashanti", "drobonso": "Ashanti", "nkenkaso": "Ashanti",
    "kaase": "Ashanti", "bekwai": "Ashanti", "agona ashanti": "Ashanti",
    "abuakwa": "Ashanti", "afamaso": "Ashanti", "nyinamponase": "Ashanti",
    "akrofrom": "Ashanti", "wamasi": "Ashanti", "tepa": "Ashanti",
    "tikrom": "Ashanti", "santasi": "Ashanti", "buokrom": "Ashanti",
    "mankranso": "Ashanti", "kokofu": "Ashanti", "kumawu": "Ashanti",
    "donyina": "Ashanti", "asamang": "Ashanti", "offinso": "Ashanti",
    "boamadumasi": "Ashanti", "jacobu": "Ashanti", "afransi": "Ashanti",
    "agogo": "Ashanti", "ahodwo": "Ashanti", "asokwa": "Ashanti",
    "kronum": "Ashanti", "kumasi metropolitan": "Ashanti",
    "asuofia": "Ashanti", "anyinasuso": "Ashanti", "anyinasusu": "Ashanti",
    "ahimakrom": "Ashanti", "apaaso": "Ashanti",
    # ── Western ──────────────────────────────────────────────────────────────
    "takoradi": "Western", "sekondi": "Western", "axim": "Western",
    "tarkwa": "Western", "half assini": "Western", "prestea": "Western",
    "bogoso": "Western", "sefwi asawinso": "Western", "sefwi essam": "Western",
    "sefwi wiawso": "Western", "enchi": "Western", "dadieso": "Western",
    "daboase": "Western", "apremdo": "Western", "aboadze": "Western",
    "kwesimintsim": "Western", "mataheko": "Western", "new takoradi": "Western",
    "manso amenfi": "Western", "dixcove": "Western", "nsuta": "Western",
    "benso": "Western", "sefwi bekwai": "Western", "sefwi boinzan": "Western",
    # ── Western North ────────────────────────────────────────────────────────
    "bibiani": "Western North", "sefwi bodi": "Western North",
    "sefwi": "Western North", "juaboso": "Western North", "anhwiaso": "Western North",
    # ── Eastern ──────────────────────────────────────────────────────────────
    "koforidua": "Eastern", "nkawkaw": "Eastern", "suhum": "Eastern",
    "somanya": "Eastern", "akyem oda": "Eastern", "kade": "Eastern",
    "asamankese": "Eastern", "mpraeso": "Eastern", "abetifi": "Eastern",
    "abomosu": "Eastern", "new abirim": "Eastern", "obosomase": "Eastern",
    "akosombo": "Eastern", "mampong-akwapim": "Eastern", "akim oda": "Eastern",
    # ── Central (v7/v8: kasoa = Central, NOT Greater Accra) ─────────────────
    "kasoa": "Central",
    "cape coast": "Central", "winneba": "Central", "saltpond": "Central",
    "swedru": "Central", "ajumako": "Central", "mumford": "Central",
    "assin fosu": "Central", "mankessim": "Central",
    "ankaful": "Central", "buduburam": "Central", "abura": "Central",
    "ayanfuri": "Central", "agona swedru": "Central",
    "agona nkwanta": "Central", "cabo corso": "Central", "asin": "Central",
    # ── Volta ────────────────────────────────────────────────────────────────
    "ho": "Volta", "hohoe": "Volta", "keta": "Volta", "akatsi": "Volta",
    "aflao": "Volta", "kpandu": "Volta", "jasikan": "Volta",
    "anloga": "Volta", "sogakope": "Volta", "adidome": "Volta",
    "battor": "Volta", "anfoega": "Volta",
    # ── Oti ──────────────────────────────────────────────────────────────────
    "dambai": "Oti", "nkwanta": "Oti", "yabologu": "Oti", "bimbila": "Northern",
    # ── Northern ─────────────────────────────────────────────────────────────
    "tamale": "Northern", "walewale": "Northern", "yendi": "Northern",
    "savelugu": "Northern", "tolon": "Northern", "saboba": "Northern",
    "kamgbunli": "Northern",
    # ── Savannah ─────────────────────────────────────────────────────────────
    "salaga": "Savannah", "damongo": "Savannah", "bole": "Savannah",
    # ── North East ───────────────────────────────────────────────────────────
    "nalerigu": "North East", "kparigu": "North East",
    # ── Bono East ────────────────────────────────────────────────────────────
    "techiman": "Bono East", "nkoranza": "Bono East", "kintampo": "Bono East",
    "atebubu": "Bono East", "yeji": "Bono East",
    # ── Brong-Ahafo ──────────────────────────────────────────────────────────
     "berekum": "Brong-Ahafo",
    "banda": "Brong-Ahafo", "wenchi": "Brong-Ahafo",
    "dormaa ahenkro": "Brong-Ahafo", "abesim": "Brong-Ahafo",
    # ── Upper East ───────────────────────────────────────────────────────────
    "bolgatanga": "Upper East", "navrongo": "Upper East", "bawku": "Upper East",
    "zebilla": "Upper East", "sandema": "Upper East",
    # ── Upper West ───────────────────────────────────────────────────────────
    "wa": "Upper West", "lawra": "Upper West", "nandom": "Upper West",
    "jirapa": "Upper West", "nadawli": "Upper West", "daffiama": "Upper West",
    "wechiau": "Upper West", "lamboya": "Upper West",
    # ── Ahafo ────────────────────────────────────────────────────────────────
    "goaso": "Ahafo", "bechem": "Ahafo", "duayaw nkwanta": "Ahafo",
    "kukuom": "Ahafo", "kenyasi": "Ahafo", "acherensua": "Ahafo",
    "adjoum": "Ahafo", "adum banso": "Ahafo",

    "elmina": "Central",
    "winneba": "Central",
    "hohoe": "Volta",  # already partially covered
    "tamale metropolis": "Northern",
    "sekondi-takoradi": "Western",
}

# v8: Text signals — kasoa → Central, Ghana NOT a signal for Greater Accra
REGION_TEXT_SIGNALS = [
    ("greater accra",    "Greater Accra"), ("accra metropolis", "Greater Accra"),
    ("accra region",     "Greater Accra"), ("accra",            "Greater Accra"),
    ("accra metropolis", "Greater Accra"),
    ("tema",             "Greater Accra"), ("ashanti",          "Ashanti"),
    ("kumasi",           "Ashanti"),       ("western region",   "Western"),
    ("takoradi",         "Western"),       ("sekondi",          "Western"),
    ("eastern region",   "Eastern"),      ("koforidua",        "Eastern"),
    ("volta region",     "Volta"),         ("central region",   "Central"),
    ("cape coast",       "Central"),
    ("kasoa",            "Central"),       # v8: hard here AND in CITY_TO_REGION
    ("northern region",  "Northern"),      ("tamale",           "Northern"),
    ("upper east",       "Upper East"),   ("bolgatanga",        "Upper East"),
    ("upper west",       "Upper West"),   (" wa ", "Upper West"),
    ("wa", "Upper West"),
    ("brong",            "Brong-Ahafo"),  ("sunyani",           "Brong-Ahafo"),
    ("bono east",        "Bono East"),    ("techiman",          "Bono East"),
    ("ahafo",            "Ahafo"),        ("goaso",             "Ahafo"),
    ("savannah",         "Savannah"),     ("damongo",           "Savannah"),
    ("north east region","North East"),   ("nalerigu",          "North East"),
    ("western north",    "Western North"),("bibiani",           "Western North"),
    ("oti region",       "Oti"),          ("dambai",            "Oti"),
    ("bono", "Bono"),
    # v8: "ghana" deliberately NOT listed as a signal for any specific region
]


def _normalise_region(state_val, city_val, name_val, desc_val, addr1, addr2, addr3):
    """4-layer region normalisation. v8: Kasoa hard override + no Ghana→Greater Accra."""

    # v8 HARD OVERRIDE: Kasoa must always be Central — checked before everything else
    city_lower_raw = str(city_val or "").strip().lower()
    if "kasoa" in city_lower_raw:
        return ("Central", "kasoa_hard_override", 1.0)

    # Also check name + address fields for kasoa
    all_text_lower = " ".join(str(x or "").lower() for x in [name_val, addr1, addr2, addr3])
    if re.search(r"\bkasoa\b", all_text_lower):
        return ("Central", "kasoa_hard_override", 0.95)

    # Layer 1: state/region field
    if state_val and str(state_val).strip() not in ("", "null", "None", "nan"):
        r = str(state_val).strip()
        # v8: Skip "Ghana" — it is NOT a valid region signal
        if r.lower() in ("ghana", "gh"):
            pass  # fall through to next layer
        else:
            mapped = REGION_NORM_MAP.get(r) or REGION_NORM_MAP.get(r.title())
            if not mapped:
                stripped = re.sub(r"\s*[Rr]egion$", "", r).strip()
                mapped = REGION_NORM_MAP.get(stripped) or REGION_NORM_MAP.get(stripped.title())
            if not mapped and r.title() in VALID_REGIONS:
                mapped = r.title()
            if mapped and mapped in VALID_REGIONS:
                return (mapped, "state_field", 1.0)

    # Layer 2: city lookup
    if city_val and str(city_val).strip() not in ("", "null", "None", "nan"):
        city_lower = str(city_val).strip().lower()
        region = CITY_TO_REGION.get(city_lower)
        if region:
            return (region, "city_lookup", 1.0)
        for city_key, reg in CITY_TO_REGION.items():
            if city_lower.startswith(city_key) or city_key.startswith(city_lower):
                return (reg, "city_lookup", 0.85)

    # Layer 3: text signal scan (Ghana NOT included as signal)
    search_text = " ".join([str(x or "").lower() for x in [name_val, desc_val, addr1, addr2, addr3]])
    for signal, region in REGION_TEXT_SIGNALS:
        if signal in search_text:
            return (region, "text_signal", 0.65)

    return ("Unknown", "unknown", 0.0)


@pandas_udf(ArrayType(StringType()))
def normalise_region_udf(
    state: pd.Series, city: pd.Series, name: pd.Series,
    desc: pd.Series, addr1: pd.Series, addr2: pd.Series, addr3: pd.Series,
) -> pd.Series:
    result = []
    for args in zip(state, city, name, desc, addr1, addr2, addr3):
        r, src, conf = _normalise_region(*args)
        result.append([r, src, str(conf)])
    return pd.Series(result)


region_result = normalise_region_udf(
    F.col("address_stateOrRegion"), F.col("address_city"), F.col("name"),
    F.col("description"), F.col("address_line1"), F.col("address_line2"), F.col("address_line3"),
)

df = df \
    .withColumn("_region_arr",       region_result) \
    .withColumn("region_normalised", F.col("_region_arr").getItem(0)) \
    .withColumn("region_source",     F.col("_region_arr").getItem(1)) \
    .withColumn("region_confidence", F.col("_region_arr").getItem(2).cast(FloatType())) \
    .drop("_region_arr")

print("Region normalisation complete (v8: Kasoa hard override + no Ghana→Greater Accra).")
df.groupBy("region_normalised").count().orderBy(F.desc("count")).show(25)
unknown_pct = df.filter(F.col("region_normalised") == "Unknown").count() / df.count() * 100
print(f"Unknown region: {unknown_pct:.1f}%")

# COMMAND ----------
# MAGIC %md ## 13. Facility Type Inference — v8: Internal extended types + PDF enum mapping

# COMMAND ----------


_FTYPE_TYPO_MAP = {
    "farmacy": "pharmacy", "pharamcy": "pharmacy", "pharmcy": "pharmacy",
    "hospitl": "hospital", "clinc": "clinic", "dentits": "dentist", "doctr": "doctor",
}

_FTYPE_RULES = [
    (re.compile(r"\b(?:teaching|referral|specialist|regional|district|municipal|general|primary|secondary|mission|CHAG|military|university|psychiatric|tertiary)\s+hospital\b", re.I), "hospital", 0.95),
    (re.compile(r"\bhospital\b", re.I), "hospital", 0.90),
    (re.compile(r"\bmedical\s+(?:complex|centre|center)\b", re.I), "hospital", 0.85),
    (re.compile(r"\bhealth\s+complex\b", re.I), "hospital", 0.80),
    (re.compile(r"\bpharmac(?:y|ies|eutical|ist)\b", re.I), "pharmacy", 0.90),
    (re.compile(r"\bchemist\b|\bdrug\s+(?:store|shop)\b", re.I), "pharmacy", 0.85),
    (re.compile(r"\b(?:licensed\s+)?chemical\s+(?:seller|shop|store)\b|\bOTCMS\b|\bLCS\b", re.I), "pharmacy", 0.82),
    (re.compile(r"\bchemical\s+shop\b", re.I), "pharmacy", 0.80),
    (re.compile(r"\bdental\s+(?:centre|center|clinic|hospital)\b", re.I), "dentist", 0.92),
    (re.compile(r"\bdental\b|\bdentist\b|\bdentistry\b|\borthodont\b|\boral\s+(?:care|health)\b", re.I), "dentist", 0.88),
    # Extended internal types
    (re.compile(r"\beye\s+(?:care|clinic|centre|center|hospital)\b|\boptical\s+(?:clinic|centre|center)\b", re.I), "eye_clinic", 0.92),
    (re.compile(r"\bophthalmolog(?:y|ist)\b|\boptometr(?:y|ist)\b", re.I), "eye_clinic", 0.88),
    (re.compile(r"\bchps\b|\bcommunity\s+health\s+planning\s+and\s+services?\b|\bcommunity\s+health\s+post\b", re.I), "chps", 0.92),
    (re.compile(r"\bdiagnostic\s+(?:cent(?:er|re)|laboratory|lab|services?)\b", re.I), "diagnostic_center", 0.88),
    (re.compile(r"\bmedical\s+laborator(?:y|ies)\b|\bmedical\s+lab\b", re.I), "diagnostic_center", 0.85),
    (re.compile(r"\bimaging\s+cent(?:er|re)\b|\bradiology\b|\bscan\s+cent", re.I), "diagnostic_center", 0.82),
    (re.compile(r"\blaborator(?:y|ies)\b", re.I), "diagnostic_center", 0.70),
    (re.compile(r"\bmaternity\s+(?:home|clinic|unit|centre|center)\b", re.I), "maternity_home", 0.92),
    (re.compile(r"\bgeneral\s+practitioner\b|\bprivate\s+practice\b|\bphysician\b|\bmedical\s+practice\b", re.I), "doctor", 0.85),
    (re.compile(r"\bdr\.?\s+[a-z]+\b", re.I), "doctor", 0.68),
    (re.compile(r"\bphysiotherapy\b|\brehabilitation\s+cent(?:er|re)\b", re.I), "clinic", 0.75),
    (re.compile(r"\bclinic\b|\bpolyclinic\b", re.I), "clinic", 0.85),
    (re.compile(r"\bcommunity\s+health\s+(?:center|centre)\b", re.I), "clinic", 0.82),
    (re.compile(r"\bhealth\s+cent(?:er|re)\b", re.I), "clinic", 0.80),
    (re.compile(r"\bhealth\s+post\b|\bhealth\s+facilit(?:y|ies)\b", re.I), "clinic", 0.75),
]

_HYBRID_NGO_CLINICAL = re.compile(
    r"\b(?:chag|health\s+service|district\s+health|regional\s+health|ghana\s+health\s+service|private\s+health)\b",
    re.I,
)



def _infer_facility_type(existing, name, description, capability_items, org_type):
    """Returns (facility_type_clean, confidence). facility_type_clean may be an extended type."""
    raw = str(existing or "").strip().lower()
    if raw and raw not in ("null", "none", ""):
        corrected = _FTYPE_TYPO_MAP.get(raw, raw)
        all_types = set(EXTENDED_TO_PDF_ENUM.keys())
        if corrected in all_types:
            # downgrade confidence unless very strong match
            if corrected in ("hospital", "pharmacy"):
                return corrected, 0.95
            return corrected, 0.85

    org_lower = str(org_type or "").strip().lower()
    name_str  = str(name or "").lower()
    desc_str  = str(description or "").lower()

    if org_lower == "ngo":
        combined_text = f"{name_str} {desc_str}"
        is_hybrid = (
            _HYBRID_NGO_CLINICAL.search(combined_text) or
            _CLINICAL_INDICATORS.search(desc_str[:200])
        )
        if not is_hybrid:
            # downgrade instead of dropping
            return None, 0.1

    cap_text  = " ".join([str(c or "") for c in (capability_items or [])]).lower()
    for text, weight in [(name_str, 1.0), (cap_text, 0.9), (desc_str, 0.8)]:
        if not text or str(text).strip().lower() in ("null", "none", ""):
            continue
        for pattern, ftype, base_conf in _FTYPE_RULES:
            if pattern.search(str(text)):
                conf = round(base_conf * weight, 3)
                if org_lower == "ngo":
                    conf = round(conf * 0.7, 3)
                return ftype, conf

    # v8: If clinical evidence exists but no name match → conservative clinic
    if _CLINICAL_INDICATORS.search(f"{name_str} {desc_str} {cap_text}"):
        conf = 0.30 if org_lower == "ngo" else 0.35
        return "clinic", conf

    if org_lower == "facility":
        return "clinic", 0.55

    return None, 0.0


@pandas_udf(ArrayType(StringType()))
def infer_facility_type_udf(
    existing: pd.Series, name: pd.Series, desc: pd.Series,
    cap: pd.Series, org: pd.Series,
) -> pd.Series:
    result = []
    for ex, nm, dc, ca, ot in zip(existing, name, desc, cap, org):
        cap_items = list(ca) if isinstance(ca, (list, tuple)) else []
        ftype, conf = _infer_facility_type(ex, nm, dc, cap_items, ot)
        result.append([ftype or "", str(conf)])
    return pd.Series(result)


ftype_result = infer_facility_type_udf(
    F.col("facilityTypeId"), F.col("name"), F.col("description"),
    F.col("capability_parsed"), F.col("organization_type"),
)

df = df \
    .withColumn("_ftype_arr", ftype_result) \
    .withColumn("facility_type_clean",
                F.when(F.col("_ftype_arr").getItem(0) == "", F.lit(None))
                 .otherwise(F.col("_ftype_arr").getItem(0))) \
    .withColumn("facility_type_confidence",
                F.col("_ftype_arr").getItem(1).cast(DoubleType())) \
    .drop("_ftype_arr")

# v8-1: Map canonical facilityTypeId back to PDF enum
# (keeps extended type in facility_type_clean as internal helper)
mapping_expr = F.create_map(
    *[F.lit(x) for x in sum([[k, v] for k, v in EXTENDED_TO_PDF_ENUM.items()], [])]
)

df = df.withColumn(
    "facilityTypeId",
    F.when(F.col("facility_type_clean").isNotNull(),
           mapping_expr[F.lower(F.col("facility_type_clean"))])
     .otherwise(F.col("facilityTypeId"))   # <-- keep original
)

print("Facility type inference (v8) complete:")
print("  facility_type_clean (internal):")
df.groupBy("facility_type_clean").count().orderBy(F.desc("count")).show()
print("  facilityTypeId (PDF enum):")
df.groupBy("facilityTypeId").count().orderBy(F.desc("count")).show()

# COMMAND ----------
# MAGIC %md ## 14. operatorTypeId Validation (v8 — PDF enum: public/private only)

# COMMAND ----------

df = df.withColumn(
    "operatorTypeId",
    F.when(F.lower(F.trim(F.col("operatorTypeId"))).isin("public", "government", "govt", "ghanaian government", "state"), F.lit("public"))
     .when(F.lower(F.trim(F.col("operatorTypeId"))).isin("private", "private sector", "privately owned"), F.lit("private"))
     .when(F.col("operatorTypeId").isNotNull() & (F.trim(F.col("operatorTypeId")) != ""), F.lit(None))  # invalid → null
     .otherwise(F.lit(None).cast(StringType()))
)

print("operatorTypeId validated to PDF enum (public/private only).")
df.groupBy("operatorTypeId").count().show()

# COMMAND ----------
# MAGIC %md ## 15. Organisation Type Clean & City Normalisation

# COMMAND ----------

df = df.withColumn(
    "organization_type_clean",
    F.when(F.lower(F.col("organization_type")) == "ngo",      F.lit("ngo"))
     .when(F.lower(F.col("organization_type")) == "facility",  F.lit("facility"))
     .when(
         F.col("missionstatement").isNotNull() &
         (F.length(F.trim(F.col("missionstatement"))) > 20) &
         F.col("organization_type").isNull(),
         F.lit("ngo")
     )
     .otherwise(F.lit(None).cast(StringType()))
)


@pandas_udf(StringType())
def normalise_city_udf(city: pd.Series, addr1: pd.Series) -> pd.Series:
    CITY_FIX = {
        "accra central": "Accra", "greater accra": "Accra",
        "sekondi-takoradi": "Takoradi", "osu – accra east": "Osu",
        "accra east": "Accra", "accra north": "Accra",
        "kumasi metropolitan": "Kumasi", "kumasi metro": "Kumasi",
        "atonsu kumasi": "Atonsu", "abesim - sunyani": "Abesim",
        "cabo corso": "Cape Coast", "cape-coast": "Cape Coast",
        "ghana": None,  # v8: "Ghana" as city → null (not a city)
    }
    result = []
    for city_val, addr1_val in zip(city, addr1):
        c = str(city_val or "").strip()
        if c.lower() in ("", "null", "none", "nan"):
            c = ""
        if not c and addr1_val:
            a = str(addr1_val or "").lower()
            for k in CITY_TO_REGION.keys():
                if k in a and len(k) > 4:
                    c = k.title(); break
        if not c:
            result.append(None); continue
        c_lower = c.lower()
        if c_lower in CITY_FIX:
            result.append(CITY_FIX[c_lower])
        else:
            c = re.sub(r"\s+(?:municipal(?:ity)?|district|metropol(?:is|itan)?|assembly)\s*$", "", c, flags=re.I)
            result.append(c.strip().title() if c.strip() else None)
    return pd.Series(result)


df = df.withColumn("city_clean", normalise_city_udf(F.col("address_city"), F.col("address_line1")))
print("Organisation type and city normalization done.")

# COMMAND ----------
# MAGIC %md ## 16. Geocoding — v7 4-tier (unchanged, Kasoa coords now correct via region fix)

# COMMAND ----------

CITY_COORDS: Dict[str, Tuple[float, float]] = {
    "Accra":       (5.6037, -0.1870), "Tema":        (5.6698, -0.0166),
    "Ashaiman":    (5.7000, -0.0333), "Madina":      (5.6833, -0.1667),
    "East Legon":  (5.6333, -0.1667), "Dome":        (5.6667, -0.2333),
    "Dansoman":    (5.5667, -0.2500), "Nungua":      (5.5833, -0.0667),
    "Teshie":      (5.5833, -0.0833), "Osu":         (5.5667, -0.1833),
    "Kumasi":      (6.6885, -1.6244), "Obuasi":      (6.2000, -1.6667),
    "Ejisu":       (6.7167, -1.5833), "Mampong":     (7.0667, -1.4000),
    "Takoradi":    (4.8845, -1.7554), "Sekondi":     (4.9333, -1.7000),
    "Axim":        (4.8678, -2.2378), "Tarkwa":      (5.3003, -1.9936),
    "Cape Coast":  (5.1053, -1.2466), "Winneba":     (5.3500, -0.6333),
    "Saltpond":    (5.2000, -1.0667), "Mankessim":   (5.2667, -1.0167),
    "Kasoa":       (5.5100, -0.4350),  # Central Region coords
    "Ho":          (6.6000,  0.4667), "Hohoe":       (7.1500,  0.4667),
    "Keta":        (5.9167,  0.9833), "Akatsi":      (6.1167,  0.8000),
    "Sogakope":    (5.8833,  0.5833), "Aflao":       (6.1167,  1.1833),
    "Tamale":      (9.4075, -0.8533), "Yendi":       (9.4417, -0.0083),
    "Walewale":    (10.3667,-0.8000), "Savelugu":    (9.6167, -0.8167),
    "Bolgatanga":  (10.7861,-0.8522), "Navrongo":    (10.9000,-1.0833),
    "Bawku":       (11.0583,-0.2417), "Zebilla":     (10.8833,-0.5167),
    "Wa":          (10.0600,-2.5000), "Lawra":       (10.6333,-2.9000),
    "Jirapa":      (10.5333,-2.7500), "Nandom":      (10.8500,-2.7500),
    "Koforidua":   (6.0886, -0.2594), "Nkawkaw":    (6.5500, -0.7667),
    "Suhum":       (6.0333, -0.4500), "Somanya":    (6.1000, -0.0167),
    "Sunyani":     (7.3333, -2.3167), "Berekum":    (7.4500, -2.5833),
    "Techiman":    (7.5833, -1.9333), "Nkoranza":   (7.6667, -1.7000),
    "Kintampo":    (8.0500, -1.7167), "Atebubu":    (7.7500, -0.9833),
    "Goaso":       (6.8000, -2.5167), "Bechem":     (7.1500, -2.3000),
    "Acherensua":  (7.0667, -2.5333),
    "Nalerigu":    (10.5167,-0.3500), "Bimbila":    (9.8667, -0.0833),
    "Damongo":     (9.0833, -1.8167), "Bole":       (9.0167, -2.4833),
    "Dambai":      (8.0667,  0.1833), "Nkwanta":   (8.6167,  0.5167),
    "Nsawam":      (5.8000, -0.3500), "Dodowa":    (5.9000, -0.1167),
    "Weija":       (5.5833, -0.3000),
    "Tesano":      (5.6167, -0.2167), "Lapaz":     (5.6167, -0.2500),
}

REGION_CENTROIDS: Dict[str, Tuple[float, float]] = {
    "Greater Accra": (5.6037, -0.1870), "Ashanti":      (6.6885, -1.6244),
    "Western":       (4.9016, -2.0214), "Eastern":      (6.1500, -0.4500),
    "Central":       (5.3600, -1.2200), "Volta":        (6.8000,  0.3000),
    "Northern":      (9.5400, -0.9000), "Upper East":   (10.7800,-1.0500),
    "Upper West":    (10.2500,-2.3200), "Brong-Ahafo":  (7.3500, -1.6300),
    "Bono East":     (7.6000, -1.0000), "Ahafo":        (6.9000, -2.3000),
    "Savannah":      (8.8000, -1.7500), "North East":   (10.5000,-0.3500),
    "Western North": (6.3000, -2.7000), "Oti":          (7.4500,  0.2000),
}

DISTRICT_CENTROIDS: Dict[str, Tuple[float, float]] = {
    "Accra Metro":              (5.5500, -0.2167),
    "Tema Metro":               (5.6698, -0.0166),
    "Ga East":                  (5.7167, -0.1667),
    "Ga East Municipal":        (5.7167, -0.1667),
    "Ga West":                  (5.6500, -0.3500),
    "Ga South":                 (5.5167, -0.2667),
    "Ledzokuku":                (5.5667, -0.1000),
    "Adenta":                   (5.7000, -0.1833),
    "Kpone Katamanso":          (5.6833, -0.0167),
    "Kumasi Metro":             (6.6885, -1.6244),
    "Ejisu-Juaben":             (6.7167, -1.5833),
    "Kwabre East":              (6.7500, -1.6500),
    "Sekondi-Takoradi Metro":   (4.9000, -1.7278),
    "Cape Coast Metro":         (5.1053, -1.2466),
    "Awutu Senya East":         (5.5100, -0.4350),
    "Ho Municipal":             (6.6000,  0.4667),
    "Tamale Metro":             (9.4075, -0.8533),
    "Bolgatanga Municipal":     (10.7861,-0.8522),
    "Wa Municipal":             (10.0600,-2.5000),
    "Sunyani Municipal":        (7.3333, -2.3167),
    "Techiman Municipal":       (7.5833, -1.9333),
}


@pandas_udf(ArrayType(StringType()))
def geocode_udf(city: pd.Series, region: pd.Series, addr_state: pd.Series) -> pd.Series:
    result = []
    for city_val, region_val, state_val in zip(city, region, addr_state):
        c = str(city_val or "").strip().title()
        if c and c in CITY_COORDS:
            lat, lon = CITY_COORDS[c]
            result.append([str(lat), str(lon), "static_city_dict", "0.90"])
            continue
        found_district = False
        for lookup in [c, str(state_val or "").strip().title()]:
            if lookup and lookup in DISTRICT_CENTROIDS:
                lat, lon = DISTRICT_CENTROIDS[lookup]
                result.append([str(lat), str(lon), "district_centroid", "0.75"])
                found_district = True
                break
        if found_district:
            continue
        r = str(region_val or "").strip()
        if r and r in REGION_CENTROIDS:
            lat, lon = REGION_CENTROIDS[r]
            result.append([str(lat), str(lon), "region_centroid", "0.50"])
            continue
        result.append([str(cfg.DEFAULT_LAT), str(cfg.DEFAULT_LON), "country_centroid", "0.15"])
    return pd.Series(result)


geo_result = geocode_udf(
    F.col("city_clean"), F.col("region_normalised"), F.col("address_stateOrRegion")
)

df = df \
    .withColumn("_geo",           geo_result) \
    .withColumn("latitude",       F.col("_geo").getItem(0).cast(DoubleType())) \
    .withColumn("longitude",      F.col("_geo").getItem(1).cast(DoubleType())) \
    .withColumn("geo_source",     F.col("_geo").getItem(2)) \
    .withColumn("geo_confidence", F.col("_geo").getItem(3).cast(DoubleType())) \
    .drop("_geo")

print("Geocoding complete:")
df.groupBy("geo_source").count().show()

# COMMAND ----------
# MAGIC %md ## 17. Content Quality Flags & Counts

# COMMAND ----------

df = df \
    .withColumn("procedure_count",  F.size(F.col("procedure_parsed"))) \
    .withColumn("equipment_count",  F.size(F.col("equipment_parsed"))) \
    .withColumn("capability_count", F.size(F.col("capability_parsed"))) \
    .withColumn("specialty_count",  F.size(F.col("specialties_parsed")))

df = df \
    .withColumn("has_procedures",
                (F.col("procedure_count") > 0)) \
    .withColumn("has_equipment",
                (F.col("equipment_count") > 0)) \
    .withColumn("has_capabilities",
                (F.col("capability_count") > 0)) \
    .withColumn("has_specialties",
                (F.col("specialty_count") > 0)) \
    .withColumn("has_description",
                F.col("description").isNotNull() & (F.length(F.trim(F.col("description"))) > 30)) \
    .withColumn("has_contact",
                (F.size(F.col("phone_numbers_parsed")) > 0) |
                (F.col("email").isNotNull() & (F.trim(F.col("email")) != "")) |
                (F.col("officialWebsite").isNotNull() & (F.trim(F.col("officialWebsite")) != "")))

print("Content flags computed.")

# COMMAND ----------
# MAGIC %md ## 18. Official Phone

# COMMAND ----------

df = df.withColumn("official_phone", extract_official_phone_udf(F.col("phone_numbers_parsed")))

# COMMAND ----------
# MAGIC %md ## 19. RAG Document Text — v7 logic (unchanged)

# COMMAND ----------

def _safe_iter(x):
    if x is None: return []
    try:    return list(x)
    except: return []


def _build_doc_text(name, city, region, ftype, specs, procs, equip, caps, desc):
    parts = []
    seen_clinical: set = set()

    if name and str(name).strip():
        parts.append(f"FACILITY: {name.strip()}")

    loc_parts = [x for x in [city, region] if x and str(x).strip() not in ("", "None", "Unknown")]
    if loc_parts:
        parts.append(f"LOCATION: {', '.join(loc_parts)}")

    if ftype and str(ftype).strip():
        parts.append(f"TYPE: {ftype.strip()}")

    def _add_section(label: str, items, limit: int = 15):
        clean = []
        for s in _safe_iter(items):
            s = str(s).strip()
            if not s or len(s) < 5:
                continue
            key = re.sub(r"[^\w]", "", s.lower())[:30]
            if key not in seen_clinical:
                seen_clinical.add(key)
                clean.append(s)
        if clean:
            parts.append(f"{label}: {', '.join(clean[:limit])}")

    _add_section("SPECIALTIES",  specs)
    _add_section("PROCEDURES",   procs)
    _add_section("EQUIPMENT",    equip)
    _add_section("CAPABILITIES", caps)

    if desc and str(desc).strip():
        desc_clean = str(desc).strip()
        if _CLINICAL_INDICATORS.search(desc_clean):
            desc_clean = desc_clean[:400]
            if len(desc_clean) == 400:
                last_period = desc_clean.rfind(".")
                if last_period > 200:
                    desc_clean = desc_clean[:last_period + 1]
            parts.append(f"DESCRIPTION: {desc_clean}")

    return "\n".join(parts)


@pandas_udf(StringType())
def build_doc_text_udf(
    name: pd.Series, city: pd.Series, region: pd.Series,
    ftype: pd.Series, specs: pd.Series, procs: pd.Series,
    equip: pd.Series, caps: pd.Series, desc: pd.Series,
) -> pd.Series:
    result = []
    for args in zip(name, city, region, ftype, specs, procs, equip, caps, desc):
        result.append(_build_doc_text(*args))
    return pd.Series(result)


df = df \
    .withColumn("doc_text",
                build_doc_text_udf(
                    F.col("name"), F.col("city_clean"), F.col("region_normalised"),
                    F.col("facility_type_clean"),
                    F.col("specialties_parsed"), F.col("procedure_parsed"),
                    F.col("equipment_parsed"), F.col("capability_parsed"),
                    F.col("description"),
                )) \
    .withColumn("doc_text_length", F.length(F.col("doc_text")))

df = df.withColumn(
    "is_rag_ready",
    (F.col("doc_text_length") > 80) &
    (
        F.col("has_procedures") |
        F.col("has_equipment") |
        F.col("has_specialties") |
        (F.col("has_capabilities") & (F.col("capability_count") >= 2))
    ) &
    (F.col("geo_source") != "country_centroid")
)

print(f"RAG-ready: {df.filter(F.col('is_rag_ready')).count():,} / {df.count():,}")

# COMMAND ----------
# MAGIC %md ## 20. Row Citations — Field-Level Provenance

# COMMAND ----------

CITATION_SCHEMA = ArrayType(StructType([
    StructField("field",             StringType(), True),
    StructField("snippet",           StringType(), True),
    StructField("source_column",     StringType(), True),
    StructField("extraction_method", StringType(), True),
    StructField("confidence",        FloatType(),  True),
]))


def _build_citations(proc, equip, cap, spec, desc,
                     proc_raw_count, equip_raw_count,
                     year_int, capacity_int, doctors_int,
                     year_src, capacity_src, doctors_src,
                     year_conf, capacity_conf, doctors_conf):
    citations = []

    def valid(x):
        return x and len(str(x).strip()) > 5

    raw_procs_count  = int(proc_raw_count  or 0)
    raw_equip_count  = int(equip_raw_count or 0)

    for item in (proc or [])[:3]:
        if valid(item):
            method, src_col, conf = (
                ("direct_json_parse", "procedure", 0.90) if raw_procs_count > 0
                else ("mined_from_description_or_capability", "description", 0.78)
            )
            citations.append({"field": "procedure_parsed", "snippet": str(item).strip()[:100],
                               "source_column": src_col, "extraction_method": method, "confidence": float(conf)})

    for item in (equip or [])[:3]:
        if valid(item):
            method, src_col, conf = (
                ("direct_json_parse", "equipment", 0.90) if raw_equip_count > 0
                else ("mined_or_context_inferred", "description", 0.72)
            )
            citations.append({"field": "equipment_parsed", "snippet": str(item).strip()[:100],
                               "source_column": src_col, "extraction_method": method, "confidence": float(conf)})

    for item in (cap or [])[:2]:
        if valid(item):
            citations.append({"field": "capability_parsed", "snippet": str(item).strip()[:100],
                               "source_column": "capability", "extraction_method": "json_parse_filtered",
                               "confidence": float(0.82)})

    for item in (spec or [])[:2]:
        if valid(item):
            citations.append({"field": "specialties_parsed", "snippet": str(item).strip()[:100],
                               "source_column": "specialties", "extraction_method": "direct_json_parse_normalized",
                               "confidence": float(0.95)})

    if year_int is not None:
        y_src = str(year_src or "").strip()
        src_col = "yearestablished" if y_src and y_src not in ("", "null", "None") else "description"
        conf_map = {"high": 0.95, "medium": 0.82, "low": 0.65, "inferred": 0.45}
        conf = conf_map.get(str(year_conf or ""), 0.70)
        citations.append({"field": "year_established_int", "snippet": str(year_int),
                           "source_column": src_col,
                           "extraction_method": f"year_extraction_{year_conf or 'unknown'}",
                           "confidence": float(conf)})

    if capacity_int is not None:
        c_src = str(capacity_src or "").strip()
        src_col = "capacity" if c_src and c_src not in ("", "null", "None") else "description"
        conf_map = {"high": 0.92, "medium": 0.78, "low": 0.60, "inferred": 0.40}
        conf = conf_map.get(str(capacity_conf or ""), 0.65)
        citations.append({"field": "capacity_int", "snippet": str(capacity_int),
                           "source_column": src_col,
                           "extraction_method": f"capacity_extraction_{capacity_conf or 'unknown'}",
                           "confidence": float(conf)})

    if doctors_int is not None:
        d_src = str(doctors_src or "").strip()
        src_col = "numberdoctors" if d_src and d_src not in ("", "null", "None") else "description"
        conf_map = {"high": 0.92, "medium": 0.78, "low": 0.60}
        conf = conf_map.get(str(doctors_conf or ""), 0.65)
        citations.append({"field": "number_doctors_int", "snippet": str(doctors_int),
                           "source_column": src_col,
                           "extraction_method": f"doctor_extraction_{doctors_conf or 'unknown'}",
                           "confidence": float(conf)})

    if not citations and desc and len(str(desc).strip()) >= 30:
        citations.append({"field": "description", "snippet": str(desc).strip()[:100],
                           "source_column": "description",
                           "extraction_method": "description_fallback", "confidence": float(0.50)})

    return citations


build_citations_udf = F.udf(_build_citations, CITATION_SCHEMA)

df = df.withColumn(
    "citations",
    build_citations_udf(
        F.col("procedure_parsed"), F.col("equipment_parsed"),
        F.col("capability_parsed"), F.col("specialties_parsed"),
        F.col("description"),
        F.col("procedure_count"), F.col("equipment_count"),
        F.col("year_established_int"), F.col("capacity_int"), F.col("number_doctors_int"),
        F.col("yearestablished"), F.col("capacity"), F.col("numberdoctors"),
        F.col("year_confidence"), F.col("capacity_confidence"), F.col("doctors_confidence"),
    )
)

# COMMAND ----------
# MAGIC %md ## 21. Trust-Based Completeness Score

# COMMAND ----------

df = df.withColumn(
    "data_completeness_score",
    (
        F.when(F.col("name").isNotNull() & (F.trim(F.col("name")) != ""), 0.15).otherwise(0.0)
        + F.when(F.col("geo_source") == "static_city_dict",  0.12)
           .when(F.col("geo_source") == "district_centroid", 0.08)
           .when(F.col("geo_source") == "region_centroid",   0.04)
           .otherwise(0.0)
        + F.when(F.col("region_normalised") != "Unknown", 0.04).otherwise(0.0)
        + F.when(
            F.col("facility_type_clean").isNotNull() & (F.col("facility_type_confidence") >= 0.80),
            0.10
          ).when(F.col("facility_type_clean").isNotNull(), 0.06)
           .otherwise(0.0)
        + F.when(F.col("specialty_count") >= 3, 0.08)
           .when(F.col("specialty_count") >= 1, 0.05)
           .otherwise(0.0)
        + F.when(F.col("procedure_count") >= 5, 0.10)
           .when(F.col("procedure_count") >= 1, 0.06)
           .otherwise(0.0)
        + F.when(F.col("equipment_count") >= 3, 0.10)
           .when(F.col("equipment_count") >= 1, 0.06)
           .otherwise(0.0)
        + F.when(F.col("capability_count") >= 3, 0.08)
           .when(F.col("capability_count") >= 1, 0.05)
           .otherwise(0.0)
        + F.when(
            F.col("has_description") & (F.length(F.trim(F.col("description"))) > 200),
            0.10
          ).when(F.col("has_description"), 0.06)
           .otherwise(0.0)
        + F.when(F.col("has_contact"), 0.07).otherwise(0.0)
        + F.when(F.col("year_established_int").isNotNull(), 0.02).otherwise(0.0)
        + F.when(F.col("capacity_int").isNotNull(),         0.02).otherwise(0.0)
        + F.when(F.col("number_doctors_int").isNotNull(),   0.01).otherwise(0.0)
    ).cast(FloatType())
)

# COMMAND ----------
# MAGIC %md ## 22. Row Quality Flags — v8: Anomaly Detection Added

# COMMAND ----------

ROW_QUALITY_FLAGS_SCHEMA = ArrayType(StringType())


def _build_quality_flags(
    name, city, state, region, ftype, ftype_internal, org_type,
    has_proc, has_equip, has_cap, has_spec,
    has_contact, completeness, cap_arr,
    year_int, capacity_int, doctors_int,
    country_code, country, lat, lon,
    addr1, addr2, addr3, geo_src,
    specialty_arr, facility_type_confidence,
):
    flags = []

    # ── HIGH severity ────────────────────────────────────────────────────────
    if not name or str(name).strip() in ("", "null", "None"):
        flags.append("HIGH:missing_name")

    if not has_proc and not has_equip and not has_cap and not has_spec:
        flags.append("HIGH:no_clinical_data")

    ftype_str          = str(ftype or "").strip().lower()
    ftype_internal_str = str(ftype_internal or "").strip().lower()
    org_str            = str(org_type or "").strip().lower()

    # v8: facilityTypeId / org_type mismatch
    if org_str == "ngo" and ftype_str in ("hospital", "clinic", "pharmacy", "dentist"):
        flags.append("HIGH:ftype_orgtype_mismatch")

    # v8 ANOMALY: Pharmacy claims ICU or ward-level capabilities
    if ftype_internal_str == "pharmacy" and cap_arr:
        icu_re = re.compile(r"\b(icu|intensive\s+care|inpatient\s+ward|operating\s+theatre|nicu)\b", re.I)
        if any(icu_re.search(str(c)) for c in cap_arr):
            flags.append("HIGH:anomaly_pharmacy_claims_icu_or_ward")

    # v8 ANOMALY: Hospital with very low capacity (when explicitly recorded)
    if ftype_internal_str == "hospital" and capacity_int is not None and capacity_int < 5:
        flags.append("HIGH:anomaly_hospital_very_low_capacity")

    # v8 ANOMALY: Implausible doctor count
    if doctors_int is not None and (doctors_int < 1 or doctors_int > 500):
        flags.append("HIGH:implausible_doctor_count")

    # v8 ANOMALY: Implausible bed capacity
    if capacity_int is not None and (capacity_int < 5 or capacity_int > 5000):
        flags.append("HIGH:implausible_bed_capacity")

    # Suspicious year
    if year_int is not None and (year_int < 1850 or year_int > 2026):
        flags.append("HIGH:suspicious_year")

    # ── MEDIUM severity ──────────────────────────────────────────────────────
    if (not city or str(city).strip() in ("", "null", "None")) and \
       (not state or str(state).strip() in ("", "null", "None")):
        flags.append("MEDIUM:missing_location")

    if str(region or "") == "Unknown":
        flags.append("MEDIUM:unknown_region")

    if not ftype_str:
        flags.append("MEDIUM:missing_facility_type")

    if not has_contact:
        flags.append("MEDIUM:missing_contact")

    ghana_known = str(country or "").strip().lower() in ("ghana", "gh")
    if ghana_known and (not country_code or str(country_code).strip() in ("", "null", "None")):
        flags.append("MEDIUM:missing_country_code")

    if region and str(region).strip() not in ("Unknown", "") and \
       country and str(country).strip().lower() not in ("ghana", "gh", ""):
        flags.append("MEDIUM:country_region_inconsistency")

    if cap_arr:
        ADDR_SENT_RE = re.compile(
            r"(?i)(located\s+at|P\.O\.\s*box|GPS\s+(?:code|address)|suite\s+\d"
            r"|tel(?:ephone)?\s*[:\-]|\+\d{7,}|http[s]?://)", re.I
        )
        if any(ADDR_SENT_RE.search(str(c)) for c in (cap_arr or [])):
            flags.append("MEDIUM:capability_contamination_risk")

    if str(geo_src or "") == "country_centroid":
        flags.append("MEDIUM:country_centroid_geocode_only")
    elif str(geo_src or "") == "region_centroid":
        flags.append("MEDIUM:region_centroid_geocode")

    addr_text = " ".join(str(x or "") for x in [addr1, addr2, addr3, city, state])
    if len(addr_text.strip()) < 10:
        flags.append("MEDIUM:very_vague_address")

    # v8 ANOMALY: Specialty–facility type mismatch
    specs = list(specialty_arr) if specialty_arr else []
    if ftype_internal_str == "pharmacy" and any(
        s in specs for s in ["generalSurgery", "cardiacSurgery", "criticalCareMedicine"]
    ):
        flags.append("MEDIUM:anomaly_specialty_facility_mismatch")

    # v8 ANOMALY: Eye clinic but no ophthalmology specialty
    if ftype_internal_str == "eye_clinic" and specs and "ophthalmology" not in specs:
        flags.append("MEDIUM:anomaly_eye_clinic_missing_ophthalmology_specialty")

    # v8: Low facility type confidence
    try:
        if facility_type_confidence is not None and float(facility_type_confidence) < 0.50 and ftype_str:
            flags.append("MEDIUM:low_facility_type_confidence")
    except Exception:
        pass

    # ── LOW severity ─────────────────────────────────────────────────────────
    if (completeness or 0.0) < 0.25:
        flags.append("LOW:low_completeness")

    if not has_proc and not has_equip and not has_spec and has_cap:
        flags.append("LOW:capability_only_no_procedures_or_equipment")

    return flags


build_quality_flags_udf = F.udf(_build_quality_flags, ROW_QUALITY_FLAGS_SCHEMA)

df = df \
    .withColumn("row_quality_flags",
                build_quality_flags_udf(
                    F.col("name"), F.col("address_city"), F.col("address_stateOrRegion"),
                    F.col("region_normalised"),
                    F.col("facilityTypeId"),          # PDF enum
                    F.col("facility_type_clean"),      # internal extended
                    F.col("organization_type_clean"),
                    F.col("has_procedures"), F.col("has_equipment"),
                    F.col("has_capabilities"), F.col("has_specialties"),
                    F.col("has_contact"), F.col("data_completeness_score"),
                    F.col("capability_parsed"),
                    F.col("year_established_int"), F.col("capacity_int"),
                    F.col("number_doctors_int"),
                    F.col("address_countryCode"), F.col("address_country"),
                    F.col("latitude"), F.col("longitude"),
                    F.col("address_line1"), F.col("address_line2"), F.col("address_line3"),
                    F.col("geo_source"),
                    F.col("specialties_parsed"),
                    F.col("facility_type_confidence"),
                )) \
    .withColumn("quality_flag_count", F.size(F.col("row_quality_flags")))

print("Quality flags (v8: anomaly detection added) applied.")
df.select(F.explode("row_quality_flags").alias("flag")) \
  .groupBy("flag").count().orderBy(F.desc("count")).show(40)

# COMMAND ----------
# MAGIC %md ## 23. Re-evaluate RAG Readiness

# COMMAND ----------

df = df.withColumn(
    "is_rag_ready",
    F.col("is_rag_ready") &
    ~F.arrays_overlap(
        F.col("row_quality_flags"),
        F.array(
            F.lit("HIGH:no_clinical_data"),
            F.lit("HIGH:missing_name"),
            F.lit("MEDIUM:capability_contamination_risk"),
            F.lit("MEDIUM:country_centroid_geocode_only"),
        )
    )
)

rag_final = df.filter(F.col("is_rag_ready")).count()
total     = df.count()
print(f"RAG-ready (post-flag gate): {rag_final:,} / {total:,} ({rag_final / total * 100:.1f}%)")

# COMMAND ----------
# MAGIC %md ## 24. Deduplication — v8: Stronger Name Normalization

# COMMAND ----------

# v8: Normalize legal suffixes, punctuation, whitespace before clustering
_LEGAL_SUFFIX_RE = re.compile(
    r"\b(?:ltd|limited|llc|inc|plc|co\b|company|enterprise|enterprises|"
    r"gh|ghana|hospital|clinic|centre|center|medical|health|care|"
    r"international|services?)\b",
    re.I
)


def _make_dedup_key(name_val, city_val, addr1_val):
    """
    v8: Stronger normalization — strips legal suffixes, facility type words,
    punctuation, double whitespace before clustering.
    """
    parts = []
    for i, v in enumerate([name_val, city_val, addr1_val]):
        s = str(v or "").strip().lower()
        # Remove punctuation
        s = re.sub(r"[^\w\s]", " ", s)
        # Remove legal suffixes (name field only)
        if i == 0:
            s = _LEGAL_SUFFIX_RE.sub(" ", s)
        # Collapse whitespace
        s = re.sub(r"\s+", " ", s).strip()
        if s and s not in ("null", "none", "nan", ""):
            parts.append(s)
    return "|".join(parts) if parts else None


make_dedup_key_udf = F.udf(_make_dedup_key, StringType())

df = df.withColumn(
    "dedup_key",
    make_dedup_key_udf(F.col("name"), F.col("city_clean"), F.col("address_line1"))
)

# v8: Enhanced survivor scoring — uses more signals
_dedup_window = Window.partitionBy("dedup_key").orderBy(
    F.desc("data_completeness_score"),
    F.asc("quality_flag_count"),
    F.desc("procedure_count"),
    F.desc("equipment_count"),
    F.desc("specialty_count"),
    F.when(F.col("geo_source") == "static_city_dict", 0)
     .when(F.col("geo_source") == "district_centroid", 1)
     .when(F.col("geo_source") == "region_centroid", 2)
     .otherwise(3).asc(),
    F.asc("row_hash"),  # stable tiebreaker
)

df = df \
    .withColumn("_dedup_rank",            F.rank().over(_dedup_window)) \
    .withColumn("is_duplicate_survivor",  F.col("_dedup_rank") == 1) \
    .drop("_dedup_rank")


@F.udf(ArrayType(StringType()))
def _add_duplicate_flag(flags, is_survivor):
    lst = list(flags) if flags else []
    if not is_survivor:
        lst = ["HIGH:duplicate_non_survivor"] + lst
    return lst

df = df.withColumn(
    "row_quality_flags",
    _add_duplicate_flag(F.col("row_quality_flags"), F.col("is_duplicate_survivor"))
).withColumn("quality_flag_count", F.size(F.col("row_quality_flags")))

df = df.withColumn(
    "row_quality_flags",
    F.when(
        F.col("facility_type_confidence") < 0.4,
        F.array_union(F.col("row_quality_flags"), F.array(F.lit("LOW:weak_facility_type")))
    ).otherwise(F.col("row_quality_flags"))
)

total_rows    = df.count()
survivors     = df.filter(F.col("is_duplicate_survivor")).count()
non_survivors = total_rows - survivors
print(f"Deduplication complete (v8: stronger name normalization):")
print(f"  Total rows         : {total_rows:,}")
print(f"  Unique survivors   : {survivors:,}")
print(f"  Non-survivors      : {non_survivors:,}  ({non_survivors/total_rows*100:.1f}%)")

# COMMAND ----------
# MAGIC %md ## 25. Canonical Field Overwrite (v8 — PDF schema expects enriched values)

# COMMAND ----------

# v8-14: Overwrite canonical bronze/silver fields with enriched values using coalesce()
# This ensures the PDF-facing fields reflect what we actually know, not sparse source values.

df = df \
    .withColumn(
        "numberDoctors",
        F.coalesce(F.col("number_doctors_int").cast(IntegerType()), F.col("numberDoctors"))
    ) \
    .withColumn(
        "yearestablished",
        F.coalesce(
            F.col("year_established_int").cast(StringType()),
            F.col("yearestablished")
        )
    ) \
    .withColumn(
        "acceptsvolunteers",
        F.coalesce(
            F.when(F.col("accepts_volunteers_bool") == True,  F.lit("true"))
             .when(F.col("accepts_volunteers_bool") == False, F.lit("false")),
            F.col("acceptsvolunteers")
        )
    )

# capacity: bronze `capacity` is STRING — overwrite with clean int string
df = df.withColumn(
    "capacity",
    F.coalesce(
        F.col("capacity_int").cast(StringType()),
        F.col("capacity")
    )
)

print("✅ Canonical fields overwritten with enriched values (coalesce pattern).")

# COMMAND ----------
# MAGIC %md ## 26. Metadata Columns

# COMMAND ----------

df = df \
    .withColumn("ingested_at",        F.to_timestamp(F.lit(datetime.now(timezone.utc).isoformat()))) \
    .withColumn("source_file",        F.lit("bronze_facilities_raw")) \
    .withColumn("dataset_version",    F.lit("v1")) \
    .withColumn("country_scope",      F.lit("Ghana")) \
    .withColumn("extraction_version", F.lit(cfg.EXTRACTION_VER)) \
    .withColumn("row_hash",
                F.when(F.col("row_hash").isNull(),
                       F.md5(F.concat_ws("|", F.col("name"), F.col("address_city"),
                                          F.col("address_line1"), F.col("source_url"))))
                 .otherwise(F.col("row_hash")))

# COMMAND ----------
# MAGIC %md ## 27. Contract Validation Block (v8 — Before Write)

# COMMAND ----------

print("=" * 70)
print("  PRE-WRITE CONTRACT VALIDATION (v8)")
print("=" * 70)

validation_errors = []

# 1. facilityTypeId — only PDF enum values
invalid_ftype = df.filter(
    F.col("facilityTypeId").isNotNull() &
    ~F.col("facilityTypeId").isin(*list(PDF_FACILITY_TYPE_ENUM))
).count()
status = "✅" if invalid_ftype == 0 else f"❌ {invalid_ftype} rows with invalid facilityTypeId"
print(f"  [1] facilityTypeId enum: {status}")
if invalid_ftype > 0:
    validation_errors.append(f"facilityTypeId enum violation: {invalid_ftype} rows")
    df.filter(F.col("facilityTypeId").isNotNull() & ~F.col("facilityTypeId").isin(*list(PDF_FACILITY_TYPE_ENUM))) \
      .select("name", "facilityTypeId").show(5)

# 2. operatorTypeId — only public/private
invalid_optype = df.filter(
    F.col("operatorTypeId").isNotNull() &
    ~F.col("operatorTypeId").isin("public", "private")
).count()
status = "✅" if invalid_optype == 0 else f"❌ {invalid_optype} rows with invalid operatorTypeId"
print(f"  [2] operatorTypeId enum: {status}")
if invalid_optype > 0:
    validation_errors.append(f"operatorTypeId enum violation: {invalid_optype} rows")

# 3. specialties_parsed — all values in FDR taxonomy
@F.udf(ArrayType(StringType()))
def find_invalid_specs(specs):
    if not specs: return []
    return [s for s in specs if s not in FDR_SPECIALTY_VOCAB]

invalid_spec_rows = df.withColumn("_bad_specs", find_invalid_specs(F.col("specialties_parsed"))) \
                      .filter(F.size(F.col("_bad_specs")) > 0).count()
status = "✅" if invalid_spec_rows == 0 else f"⚠️  {invalid_spec_rows} rows with non-canonical specialties"
print(f"  [3] Specialty FDR taxonomy: {status}")
if invalid_spec_rows > 0:
    validation_errors.append(f"Non-canonical specialties: {invalid_spec_rows} rows")

# 4. address_countryCode — must be GH when address_country = Ghana
missing_cc = df.filter(
    (F.col("address_country") == "Ghana") &
    F.col("address_countryCode").isNull()
).count()
status = "✅" if missing_cc == 0 else f"⚠️  {missing_cc} Ghana rows missing countryCode"
print(f"  [4] address_countryCode (GH enforcement): {status}")

# 5. countries_parsed — ISO alpha-2 only
@F.udf(BooleanType())
def has_non_iso2(countries):
    if not countries: return False
    return any(c not in _VALID_ISO2 for c in countries)

non_iso2 = df.filter(has_non_iso2(F.col("countries_parsed"))).count()
status = "✅" if non_iso2 == 0 else f"⚠️  {non_iso2} rows with non-ISO alpha-2 country codes"
print(f"  [5] countries_parsed ISO alpha-2: {status}")

# 6. officialWebsite — no full URLs, domain only
invalid_web = df.filter(
    F.col("officialWebsite").isNotNull() &
    (F.col("officialWebsite").startswith("http://") | F.col("officialWebsite").startswith("https://"))
).count()
status = "✅" if invalid_web == 0 else f"⚠️  {invalid_web} officialWebsite with full URL (should be domain only)"
print(f"  [6] officialWebsite domain-only: {status}")

# 7. official_phone — all E.164
invalid_phone = df.filter(
    F.col("official_phone").isNotNull() &
    ~F.col("official_phone").startswith("+")
).count()
status = "✅" if invalid_phone == 0 else f"❌ {invalid_phone} official_phone not in E.164"
print(f"  [7] official_phone E.164: {status}")
if invalid_phone > 0:
    validation_errors.append(f"E.164 violation: {invalid_phone} rows")

# 8. Kasoa → Central (no stragglers)
kasoa_wrong = df.filter(
    (F.lower(F.col("city_clean")) == "kasoa") &
    (F.col("region_normalised") != "Central")
).count()
status = "✅" if kasoa_wrong == 0 else f"❌ {kasoa_wrong} Kasoa rows NOT mapped to Central"
print(f"  [8] Kasoa → Central: {status}")
if kasoa_wrong > 0:
    validation_errors.append(f"Kasoa misclassification: {kasoa_wrong} rows")
    df.filter((F.lower(F.col("city_clean")) == "kasoa") & (F.col("region_normalised") != "Central")) \
      .select("name", "city_clean", "region_normalised").show(5)

# 9. No Ghana→Greater Accra via country name only
gh_to_accra = df.filter(
    (F.col("region_source") == "state_field") &
    (F.col("region_normalised") == "Greater Accra") &
    (F.lower(F.col("address_stateOrRegion")).isin("ghana", "gh"))
).count()
status = "✅" if gh_to_accra == 0 else f"⚠️  {gh_to_accra} rows still using Ghana→Greater Accra fallback"
print(f"  [9] No Ghana→Greater Accra fallback: {status}")

# 10. organizationdescription — check subjective language removed
if "organizationdescription" in df.columns:
    subj_remaining = df.filter(
        F.col("organizationdescription").isNotNull() &
        F.col("organizationdescription").rlike(
            r"(?i)\b(blessed|holy|god|lord|amazing|outstanding|world-class)\b"
        )
    ).count()
    status = "✅" if subj_remaining == 0 else f"⚠️  {subj_remaining} rows with remaining subjective language"
    print(f"  [10] organizationdescription neutrality: {status}")

print()
if validation_errors:
    print(f"⚠️  {len(validation_errors)} contract violations detected:")
    for e in validation_errors:
        print(f"     - {e}")
else:
    print("✅  All contract checks passed. Safe to write.")

print("=" * 70)

# COMMAND ----------
# MAGIC %md ## 28. Final Column Selection

# COMMAND ----------

SILVER_COLUMNS = [
    "unique_id", "source_url", "name", "pk_unique_id", "mongo_db",
    "specialties", "procedure", "equipment", "capability",
    "organization_type", "content_table_id",
    "phone_numbers",   # v8: now E.164-normalized JSON string
    "email", "websites", "officialWebsite",
    "yearestablished",      # v8: overwritten with enriched value
    "acceptsvolunteers",    # v8: overwritten with enriched value
    "facebooklink", "twitterlink", "linkedinlink", "instagramlink", "logo",
    "address_line1", "address_line2", "address_line3",
    "address_city", "address_stateOrRegion", "address_zipOrPostcode",
    "address_country", "address_countryCode",
    "countries",
    "missionstatement", "missionstatementlink",
    "organizationdescription",  # v8: neutrality pass applied
    "facilityTypeId",    # v8: PDF enum only (hospital/pharmacy/doctor/clinic/dentist)
    "operatorTypeId",    # v8: PDF enum only (public/private)
    "affiliationtypeids",
    "description", "area",
    "numberDoctors",     # v8: overwritten with enriched int
    "capacity",          # v8: overwritten with enriched string
    "ingested_at", "source_file", "dataset_version", "country_scope", "row_hash",
    "specialties_parsed",       # FDR canonical camelCase normalized
    "procedure_parsed", "equipment_parsed", "capability_parsed",
    "phone_numbers_parsed",     # E.164 normalized array
    "affiliationtypeids_parsed", "countries_parsed",  # ISO alpha-2 validated
    "websites_parsed",
    "official_phone",           # E.164
    "area_int",
    "year_established_int",     # with confidence
    "year_confidence",          # v8 new
    "accepts_volunteers_bool",
    "pk_unique_id_int",
    "facility_type_clean",      # internal extended type (chps/eye_clinic/etc.)
    "facility_type_confidence",
    "organization_type_clean",
    "city_clean",
    "capacity_int",             # with confidence
    "capacity_confidence",      # v8 new
    "number_doctors_int",       # with confidence
    "doctors_confidence",       # v8 new
    "region_normalised", "region_source", "region_confidence",
    "latitude", "longitude", "geo_source", "geo_confidence",
    "has_procedures", "has_equipment", "has_capabilities", "has_specialties",
    "has_description", "has_contact",
    "procedure_count", "equipment_count", "capability_count", "specialty_count",
    "doc_text", "doc_text_length", "is_rag_ready",
    "citations", "row_quality_flags", "quality_flag_count",
    "data_completeness_score", "extraction_version",
    "dedup_key", "is_duplicate_survivor",
]

available  = set(df.columns)
missing_c  = [c for c in SILVER_COLUMNS if c not in available]
if missing_c:
    print(f"⚠️  Missing from DataFrame (will be skipped): {missing_c}")

final_cols = [c for c in SILVER_COLUMNS if c in available]
df_silver  = df.select(*final_cols)

print(f"Final silver columns : {len(final_cols)}")
print(f"Final silver rows    : {df_silver.count():,}")
df_silver.printSchema()

# COMMAND ----------
# MAGIC %md ## 29. Write Silver Table

# COMMAND ----------

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(cfg.SILVER)
)

print(f"✅  Silver table written: {cfg.SILVER}")
print(f"    Rows: {spark.table(cfg.SILVER).count():,}")

# COMMAND ----------
# MAGIC %md ## 30. Post-Write Validation & Quality Report

# COMMAND ----------

sv    = spark.table(cfg.SILVER)
total = sv.count()

print("=" * 70)
print(f"  SILVER LAYER v8 — POST-WRITE VALIDATION REPORT")
print(f"  Table : {cfg.SILVER}")
print(f"  Rows  : {total:,}")
print("=" * 70)

print("\n[1] facilityTypeId PDF enum compliance")
sv.groupBy("facilityTypeId").count().orderBy(F.desc("count")).show()
invalid = sv.filter(
    F.col("facilityTypeId").isNotNull() &
    ~F.col("facilityTypeId").isin(*list(PDF_FACILITY_TYPE_ENUM))
).count()
print(f"    Invalid facilityTypeId values: {invalid}  {'✅' if invalid == 0 else '❌'}")

print("\n[2] facility_type_clean (internal extended types)")
sv.groupBy("facility_type_clean").count().orderBy(F.desc("count")).show()

print("\n[3] operatorTypeId PDF enum compliance")
sv.groupBy("operatorTypeId").count().show()

print("\n[4] Kasoa → Central fix")
kasoa = sv.filter(F.lower(F.coalesce(F.col("city_clean"), F.lit(""))).contains("kasoa"))
print(f"    Kasoa rows: {kasoa.count()}")
wrong = kasoa.filter(F.col("region_normalised") != "Central").count()
print(f"    Still not Central: {wrong}  {'✅' if wrong == 0 else '❌'}")

print("\n[5] Ghana→Greater Accra fallback eliminated")
gh_accra = sv.filter(
    (F.col("region_normalised") == "Greater Accra") &
    (F.col("region_source") == "state_field") &
    (F.lower(F.col("address_stateOrRegion")).isin("ghana", "gh"))
).count()
print(f"    Rows using Ghana→Greater Accra: {gh_accra}  {'✅' if gh_accra == 0 else '⚠️'}")

print("\n[6] Phone normalization")
phones_total = sv.filter(F.col("official_phone").isNotNull()).count()
phones_e164  = sv.filter(F.col("official_phone").startswith("+")).count()
print(f"    With official_phone: {phones_total:,}")
print(f"    Valid E.164        : {phones_e164:,}  ({'✅' if phones_e164 == phones_total else '❌'})")

print("\n[7] Numeric field coverage + confidence breakdown")
for c in ["number_doctors_int", "capacity_int", "year_established_int", "area_int"]:
    cnt = sv.filter(F.col(c).isNotNull()).count()
    print(f"    {c:<28} {cnt:>5,}  ({cnt/total*100:.1f}%)")

print("\n    capacity_confidence breakdown:")
sv.filter(F.col("capacity_int").isNotNull()).groupBy("capacity_confidence").count().show()

print("\n[8] Specialty taxonomy compliance")
total_spec_rows = sv.filter(F.size(F.col("specialties_parsed")) > 0).count()
print(f"    Rows with specialties: {total_spec_rows:,}")

print("\n[9] Equipment coverage improvement")
equip_ct = sv.filter(F.col("has_equipment")).count()
print(f"    has_equipment = TRUE: {equip_ct:,}  ({equip_ct/total*100:.1f}%)")

print("\n[10] Anomaly flags")
for flag in [
    "HIGH:anomaly_pharmacy_claims_icu_or_ward",
    "HIGH:anomaly_hospital_very_low_capacity",
    "MEDIUM:anomaly_specialty_facility_mismatch",
    "MEDIUM:anomaly_eye_clinic_missing_ophthalmology_specialty",
    "HIGH:ftype_orgtype_mismatch",
]:
    ct = sv.filter(F.array_contains(F.col("row_quality_flags"), flag)).count()
    if ct > 0:
        print(f"    {flag}: {ct}")

print("\n[11] Canonical field overwrite validation")
yr_filled  = sv.filter(F.col("yearestablished").isNotNull()).count()
cap_filled = sv.filter(F.col("capacity").isNotNull()).count()
nd_filled  = sv.filter(F.col("numberDoctors").isNotNull()).count()
print(f"    yearestablished non-null  : {yr_filled:,}  ({yr_filled/total*100:.1f}%)")
print(f"    capacity non-null         : {cap_filled:,}  ({cap_filled/total*100:.1f}%)")
print(f"    numberDoctors non-null    : {nd_filled:,}  ({nd_filled/total*100:.1f}%)")

print("\n[12] Region coverage")
sv.groupBy("region_normalised").count().orderBy(F.desc("count")).show(20)
unknown_r = sv.filter(F.col("region_normalised") == "Unknown").count()
print(f"    Unknown region: {unknown_r:,}  ({unknown_r/total*100:.1f}%)")

print("\n[13] Deduplication summary")
survivors_ct = sv.filter(F.col("is_duplicate_survivor")).count()
print(f"    Unique survivors   : {survivors_ct:,}")
print(f"    Non-survivors      : {total - survivors_ct:,}  ({(total-survivors_ct)/total*100:.1f}%)")

print("\n[14] Completeness distribution")
sv.select(
    F.when(F.col("data_completeness_score") >= 0.75, "A: ≥0.75")
     .when(F.col("data_completeness_score") >= 0.50, "B: 0.50–0.75")
     .when(F.col("data_completeness_score") >= 0.25, "C: 0.25–0.50")
     .otherwise("D: <0.25").alias("band")
).groupBy("band").count().orderBy("band").show()

print("\n[15] RAG readiness")
rag_ct = sv.filter(F.col("is_rag_ready")).count()
print(f"    RAG-ready: {rag_ct:,}  ({rag_ct/total*100:.1f}%)")

print("\n[16] Schema contract — PDF canonical fields")
PDF_CANONICAL = [
    "official_phone", "year_established_int", "accepts_volunteers_bool",
    "facilityTypeId", "operatorTypeId", "affiliationtypeids",
    "organizationdescription", "yearestablished", "acceptsvolunteers", "officialWebsite",
    "year_confidence", "capacity_confidence", "doctors_confidence",  # v8 new
]
silver_cols = set(sv.columns)
for field in PDF_CANONICAL:
    status = "✅" if field in silver_cols else "❌ MISSING"
    print(f"    {status}  {field}")

print(f"\n{'=' * 70}")
print(f"  Silver v8 validation complete.")
print(f"  Extraction version : {cfg.EXTRACTION_VER}")
print(f"  Run timestamp      : {datetime.now(timezone.utc).isoformat()}")
print(f"{'=' * 70}")

# COMMAND ----------
# MAGIC %md ## 31. Optimise Delta Table

# # COMMAND ----------

# spark.sql(f"SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false")
# spark.sql(f"ANALYZE TABLE {cfg.SILVER} COMPUTE STATISTICS FOR COLUMNS region_normalised, facility_type_clean")
# spark.sql(f"OPTIMIZE {cfg.SILVER} ZORDER BY (region_normalised, facility_type_clean);")
# print(f"✅  OPTIMIZE + ANALYZE complete on {cfg.SILVER}")

# COMMAND ----------
# MAGIC %md ## 32. Sample Inspection

# COMMAND ----------

print("=== High-completeness survivor samples ===")
sv.filter(
    F.col("is_duplicate_survivor") &
    (F.col("data_completeness_score") >= 0.60) &
    F.col("has_procedures") &
    F.col("has_contact")
).select(
    "name", "city_clean", "region_normalised",
    "facility_type_clean", "facilityTypeId",
    "official_phone", "procedure_count", "equipment_count",
    "year_established_int", "capacity_int", "capacity_confidence",
    "data_completeness_score", "geo_source", "is_rag_ready",
).orderBy(F.desc("data_completeness_score")).show(10, truncate=60)

print("\n=== Anomaly rows ===")
sv.filter(
    F.arrays_overlap(F.col("row_quality_flags"), F.array(
        F.lit("HIGH:anomaly_pharmacy_claims_icu_or_ward"),
        F.lit("HIGH:anomaly_hospital_very_low_capacity"),
        F.lit("MEDIUM:anomaly_specialty_facility_mismatch"),
    ))
).select("name", "facility_type_clean", "capacity_int", "row_quality_flags").show(10, truncate=80)

print("\n=== New facility types (internal) ===")
sv.filter(
    F.col("facility_type_clean").isin("chps", "maternity_home", "eye_clinic", "diagnostic_center")
).select(
    "name", "facility_type_clean", "facilityTypeId",
    "facility_type_confidence", "city_clean", "region_normalised",
).show(15, truncate=60)

print("\n=== v8 complete ✅ ===")

In [0]:
%sql

Select * from virtue_foundation.ghana.silver_facilities_cleaned

In [0]:
%sql
SHOW CREATE TABLE virtue_foundation.ghana.silver_facilities_cleaned